# VisDrone → YOLO26n + ByteTrack → 21 đặc trưng → nhãn Re-ID → XGBoost

Notebook huấn luyện trên **Kaggle GPU T4 × 2**, bao phủ **Bước 2 + Bước 3** của báo cáo:
phát hiện người bằng YOLO26n, theo dõi bằng ByteTrack, trích 21 đặc trưng runtime,
sinh nhãn bằng tấn công Person Re-ID độc lập, rồi huấn luyện XGBoost dự đoán rủi ro rò rỉ.

```text
VisDrone VID (7 sequence, target_id)
        │
        ├─ YOLO26n (nc=1 person)  ──►  yolo26s.pt / best.onnx
        │
        ├─ ByteTrack              ──►  tracks_frame.csv
        │
        ├─ 21 runtime features    ──►  features_frame.csv
        │
        ├─ blur/pixelate + Re-ID attack (OSNet/ResNet50)
        │        └─►  attack_results.csv
        │
        ├─ τ chọn trên val (Youden J)  ──►  tau.json
        │
        ├─ gán nhãn track-level   ──►  risk_dataset.csv (21 features + label)
        │
        └─ XGBoost (GroupKFold theo sequence)  ──►  risk_model.json
```

## Bản đồ cell

| Cell | Việc | Vào | Ra |
|---|---|---|---|
| 1 | Setup + kiểm tra GPU/RAM/disk | – | – |
| 2 | **CONFIG**: đường dẫn + siêu tham số | – | – |
| 3 | Chuẩn hoá dataset → YOLO person + index + GT | `DATA_ROOT` | `privacy_build/<split>`, `data_person.yaml`, `sequence_index.csv`, `gt_boxes.csv` |
| 4 | Audit dataset + thống kê identity | `privacy_build/<split>` + GT | `dataset_audit.json`, `identity_stats.json` |
| 5 | Train YOLO26n person (DDP + OOM ladder) | `data_person.yaml` | `runs/<RUN_NAME>/weights/yolo26s.pt` |
| 6 | Eval detector + sweep conf/iou | `yolo26s.pt` | `det_metrics.json`, `threshold_sweep.csv` |
| 7 | ByteTrack trên train/val/test | `yolo26s.pt` + index | `tracks_frame.csv` |
| 8 | Trích **21 đặc trưng** mỗi frame | `tracks_frame.csv` | `features_frame.csv`, `feature_schema.json` |
| 9 | Ẩn danh + tấn công Re-ID độc lập | crops + GT identity | `samples.csv`, `attack_results.csv` |
| 10 | Chọn ngưỡng τ trên val (Youden J) | `attack_results.csv` | `tau.json`, `roc.png` |
| 11 | Gán nhãn + gộp track-level | features + attack | **`risk_dataset.csv`** |
| 12 | Train XGBoost (GroupKFold theo sequence) | `risk_dataset.csv` | `risk_model.json`, `risk_oof.csv` |
| 13 | Đo hiệu năng + SHAP + export | model + OOF | `risk_metrics.json`, plots, zip |

## Quy trình đánh nhãn (5 bước)

| Bước báo cáo | Triển khai trong notebook |
|---|---|
| 1. Chuẩn bị video gốc + ID | `target_id` của VisDrone VID → identity = `(sequence, target_id)` |
| 2. Chạy hệ thống ẩn danh | `cv2.GaussianBlur` hoặc pixelation trên đúng box mà ByteTrack xuất ra |
| 3. Mô hình tấn công độc lập | OSNet (torchreid) nếu có, fallback ResNet50-ImageNet |
| 4. Gán nhãn từ kết quả tấn công | `1` nếu top-1 đúng **và** `cos_sim ≥ τ`; `0` nếu sai/không đạt; `NaN` nếu thiếu dữ liệu |
| 5. Huấn luyện mô hình rủi ro | XGBoost trên 21 đặc trưng đã gộp track-level |

**Chống tấn công tầm thường:** gallery lấy từ cùng sequence nhưng cách probe ≥ `GALLERY_MIN_GAP` frame,
không bao giờ lấy từ chính cửa sổ đang kiểm tra; chia tập theo sequence (`GroupKFold`), không chia theo frame.

Track quá ngắn (không đủ frame cách xa cửa sổ) sẽ không có gallery → nhãn `NaN`, đúng quy ước "không đủ bằng chứng".
Nếu số mẫu có nhãn quá ít, giảm `GALLERY_MIN_GAP` xuống 30 hoặc đặt `WINDOW_SIZE = 0`.

**Ngưỡng τ:** chọn trên `val` bằng Youden's J, đóng băng rồi mới áp cho `test`.

## 21 đặc trưng (thứ tự chuẩn)

| # | Tên | Nhóm | Công thức |
|---|---|---|---|
| 1 | `conf` | Detection | `c_t` |
| 2 | `conf_std5` | Detection | `std(c_{t-4..t})` |
| 3 | `conf_delta` | Detection | `c_t − c_{t−1}` |
| 4 | `track_age` | Tracker | `t − t_start + 1` |
| 5 | `hit_ratio` | Tracker | `N_matched / N_observed` |
| 6 | `time_since_update` | Tracker | `t − t_last_update` |
| 7 | `track_state` | Tracker | `0 tentative, 1 tracked, 2 lost` |
| 8 | `min_side_px` | Geometry | `min(w, h)` |
| 9 | `aspect_dev` | Geometry | `\|w/h − r_ref\|` |
| 10 | `dist_edge` | Geometry | `min(x1, y1, W−x2, H−y2) / D` |
| 11 | `is_trunc` | Geometry | `1` nếu chạm biên ±2 px |
| 12 | `speed_norm` | Motion | `‖p_t − p_{t−1}‖ / D` |
| 13 | `pred_uncert` | Motion | proxy `std(center_{t−4..t}) / D` |
| 14 | `lap_var_log` | Quality | `log1p(Var(∇²gray_crop))` |
| 15 | `brightness` | Quality | `mean(gray_crop) / 255` |
| 16 | `motion_blur_ratio` | Quality | `(L_{t−1} − L_t) / (L_{t−1} + L_t + ε)` |
| 17 | `nearest_dist_norm` | Scene | `min_j ‖p_i − p_j‖ / D` |
| 18 | `n_neighbors_r2_norm` | Scene | `min(N_i(r) / K, 1)` |
| 19 | `overlap_max` | Scene | `max_j IoU(B_i, B_j)` |
| 20 | `frame_det_delta` | Global | `(N_t − N_{t−1}) / (N_{t−1} + 1)` |
| 21 | `n_det_expected_ratio` | Global | `N_t / (EWMA(N) + 1)`, chỉ dùng quá khứ |

Hai điểm kỹ thuật được ghi rõ trong `feature_schema.json`:

- `pred_uncert`: Ultralytics ByteTrack không expose ma trận hiệp phương sai Kalman, nên dùng proxy
  độ lệch chuẩn tâm box trên `PRED_UNCERT_WINDOW` frame gần nhất — không phải covariance thật.
- `track_state`: mã hoá ordinal để giữ đúng 21 cột; trạng thái được tái dựng từ chuỗi frame mà
  ByteTrack xuất ra (0 = frame đầu, 1 = có cập nhật, 2 = đang bị mất dấu).

## Đường dẫn dữ liệu

Mỗi thư mục split nhận 3 kiểu cấu trúc:

```text
1) <DIR>/annotations + <DIR>/sequences      <- VisDrone VID gốc (có target_id)
2) <DIR>/images + <DIR>/labels              <- YOLO đã convert (không có target_id)
3) <DIR>/<mot-thu-muc-con>/...              <- lồng 1 cấp
```

Mặc định dùng **3 thư mục riêng** (`USE_SEPARATE_DIRS = True`) với `TRAIN_DIR`, `VAL_DIR`,
`TEST_DIR`. Nếu chỉ có 1 thư mục gốc, đặt `USE_SEPARATE_DIRS = False` và sửa `DATA_ROOT` —
notebook tự chia 4/1/2 sequence.

Ảnh được **copy thật** về `/kaggle/working` (không symlink) vì `/kaggle/input` là mount FUSE
chỉ đạt ~1 MB/s, sẽ làm mỗi epoch chậm gấp 5–10 lần.

Bật **GPU T4 × 2** và **Internet** trong Kaggle Settings. Thời gian chạy ước tính 1.5–3 h.


In [1]:
# Kaggle bootstrap: keep notebook runnable when helper .py files were not uploaded.
import json
from pathlib import Path
import numpy as np
import pandas as pd
FEATURES = ("conf", "conf_std5", "conf_delta", "track_age", "hit_ratio", "time_since_update", "track_state", "min_side_px", "aspect_dev", "dist_edge", "is_trunc", "speed_norm", "pred_uncert", "lap_var_log", "brightness", "motion_blur_ratio", "nearest_dist_norm", "n_neighbors_r2_norm", "overlap_max", "frame_det_delta", "n_det_expected_ratio")
def add_validation_split(index, seed=42, validation_fraction=0.20):
    result = index.copy()
    if "val" in set(result["split"].astype(str)):
        return result
    sequences = sorted(result.loc[result["split"].eq("train"), "sequence"].unique())
    if len(sequences) < 2:
        raise ValueError("at least two train sequences are required")
    rng = np.random.default_rng(seed); rng.shuffle(sequences)
    chosen = set(sequences[:max(1, int(round(len(sequences) * validation_fraction)))])
    result.loc[result["split"].eq("train") & result["sequence"].isin(chosen), "split"] = "val"
    return result
def repair_tracker_features(df):
    result = df.copy()
    result["time_since_update"] = pd.to_numeric(result["time_since_update"], errors="coerce").clip(lower=0).fillna(0)
    result["hit_ratio"] = pd.to_numeric(result["hit_ratio"], errors="coerce").clip(0, 1)
    result["track_state"] = pd.to_numeric(result["track_state"], errors="coerce").fillna(2).clip(0, 2)
    return result
def save_model_metadata(model_path, feature_names=FEATURES):
    if tuple(feature_names) != tuple(FEATURES): raise ValueError("feature order mismatch")
    path = Path(model_path).with_suffix(Path(model_path).suffix + ".metadata.json")
    path.write_text(json.dumps({"feature_names": list(FEATURES), "feature_count": 21}, indent=2), encoding="utf-8")
    return path
def load_model_metadata(model_path):
    path = Path(model_path).with_suffix(Path(model_path).suffix + ".metadata.json")
    data = json.loads(path.read_text(encoding="utf-8"))
    if tuple(data.get("feature_names", ())) != tuple(FEATURES): raise ValueError("feature metadata mismatch")
    return tuple(FEATURES)


In [2]:
# ── CELL 1/13 · Setup + kiểm tra phần cứng ─────────────────
# Chỉ cài gói còn THIẾU. Không dùng -U để tránh pip gỡ bản có sẵn rồi tải lỗi
# (Kaggle: pip upgrade = uninstall trước, mất mạng giữa chừng là mất luôn gói).

import importlib.util
import subprocess
import sys

REQUIRED = {
    'ultralytics': 'ultralytics',
    'xgboost': 'xgboost',
    'shap': 'shap',
    'sklearn': 'scikit-learn',
    'pandas': 'pandas',
    'yaml': 'pyyaml',
    'onnx': 'onnx',
    'onnxruntime': 'onnxruntime',
    'matplotlib': 'matplotlib',
    'cv2': 'opencv-python-headless',
    'torch': 'torch',
}

_missing = [pkg for mod, pkg in REQUIRED.items() if importlib.util.find_spec(mod) is None]
if _missing:
    print('gói còn thiếu:', _missing)
    _r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *_missing],
                        capture_output=True, text=True)
    if _r.returncode != 0:
        raise RuntimeError(
            'pip install thất bại. Bật Kaggle Settings → Internet: On rồi chạy lại cell này.\n'
            + (_r.stderr or '')[-1500:]
        )
else:
    print('tất cả gói đã có sẵn, không cần cài.')

# torchreid là tuỳ chọn (Re-ID dùng OSNet). Thiếu thì cell 9 tự fallback ResNet50.
if importlib.util.find_spec('torchreid') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'torchreid'],
                   capture_output=True, text=True)

import os
for _k in ('OMP_NUM_THREADS', 'MKL_NUM_THREADS', 'OPENBLAS_NUM_THREADS', 'NUMEXPR_NUM_THREADS'):
    os.environ[_k] = '1'
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_MODULE_LOADING'] = 'LAZY'

import csv
import gc
import json
import math
import random
import re
import shutil
import sys
import time
import warnings
from collections import Counter, defaultdict
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import torch
import ultralytics
from ultralytics import YOLO

warnings.filterwarnings('ignore')
torch.set_num_threads(1)
torch.backends.cudnn.benchmark = True

print('python     :', sys.version.split()[0])
print('ultralytics:', ultralytics.__version__)
print('torch      :', torch.__version__, '| cuda:', torch.cuda.is_available())

n_gpu = torch.cuda.device_count()
print('GPU count  :', n_gpu)
for i in range(n_gpu):
    p = torch.cuda.get_device_properties(i)
    print(f'  GPU {i}: {p.name} | {p.total_memory / 2**30:.1f} GiB')

try:
    import psutil
    vm = psutil.virtual_memory()
    print(f'RAM        : {vm.total / 2**30:.1f} GiB total | {vm.available / 2**30:.1f} GiB available')
except ImportError:
    pass

for _p in ('/kaggle/working', '/tmp', '/kaggle/input'):
    if os.path.isdir(_p):
        u = shutil.disk_usage(_p)
        print(f'disk {_p:<16}: {u.free / 2**30:.1f} GiB free / {u.total / 2**30:.1f} GiB')

if n_gpu < 2:
    print('CẢNH BÁO: chỉ thấy', n_gpu, 'GPU. Vẫn chạy được nhưng chậm hơn.')
assert torch.cuda.is_available(), 'Bật GPU T4 x2 trong Kaggle Settings trước khi chạy.'


gói còn thiếu: ['ultralytics', 'onnxruntime']
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/usage/settings
python     : 3.13.15
ultralytics: 8.4.197
torch      : 2.11.0+cu128 | cuda: True
GPU count  : 2
  GPU 0: Tesla T4 | 14.6 GiB
  GPU 1: Tesla T4 | 14.6 GiB
RAM        : 31.3 GiB total | 29.5 GiB available
disk /kaggle/working : 19.5 GiB free / 19.5 GiB
disk /tmp            : 1048.7 GiB free / 8156.9 GiB
disk /kaggle/input   : 19.5 GiB free / 19.5 GiB


In [ ]:
# ── CELL 2/13 · CONFIG ─────────────────────────────────────
# Chỉ sửa cell này.

from pathlib import Path

# ===== DỮ LIỆU =====
# Cách 1: 1 thư mục gốc, notebook tự chia sequence theo SPLIT_RATIOS.
DATA_ROOT = Path('/kaggle/input/visdrone-vid')

# Cách 2 (đang dùng): 3 thư mục riêng cho train/val/test.
USE_SEPARATE_DIRS = True
TRAIN_DIR = Path('/kaggle/input/datasets/mtu2006/visdrone-train/VisDrone2019-VID-train')
VAL_DIR   = Path('/kaggle/input/datasets/mtu2006/visdrone-val/VisDrone2019-VID-val')
TEST_DIR  = Path('/kaggle/input/datasets/mtu2006/visdrone-test/VisDrone2019-VID-test-dev')

SPLIT_RATIOS = (4, 1, 2)
SPLIT_SEED = 42

# ===== THƯ MỤC =====
WORK = Path('/kaggle/working')
BUILD_DIR = WORK / 'privacy_build'
RUNS_DIR = WORK / 'runs'
ARTIFACT = WORK / 'artifacts'

SEQ_INDEX = WORK / 'sequence_index.csv'
GT_CSV = WORK / 'gt_boxes.csv'
DATA_YAML = WORK / 'data_person.yaml'
LAYOUT_JSON = WORK / 'layout.json'

TRACKS_CSV = ARTIFACT / 'tracks_frame.csv'
FEATURES_CSV = ARTIFACT / 'features_frame.csv'
SAMPLES_CSV = ARTIFACT / 'samples.csv'
ATTACK_CSV = ARTIFACT / 'attack_results.csv'
TAU_JSON = ARTIFACT / 'tau.json'
RISK_CSV = ARTIFACT / 'risk_dataset.csv'
SCHEMA_JSON = ARTIFACT / 'feature_schema.json'
RISK_MODEL_JSON = ARTIFACT / 'risk_model.json'
RISK_OOF_CSV = ARTIFACT / 'risk_oof.csv'
RISK_METRICS_JSON = ARTIFACT / 'risk_metrics.json'
DET_METRICS_JSON = ARTIFACT / 'det_metrics.json'

# ===== DETECTOR =====
MODEL_NAME = 'yolo26s.pt'
FALLBACK = ['yolo11s.pt', 'yolo26n.pt', 'yolo11n.pt']
PERSON_CATS = (1, 2)            # VisDrone: 1=pedestrian, 2=people
IMGSZ = 768
BATCH = 8
EPOCHS = 3
WORKERS = 4
DEVICE = '0,1'
DEVICE_EVAL = 0
SEED = 42
PATIENCE = 15
RUN_NAME = 'yolo26s_visdrone_recall_t4x2'
MAX_DET = 1000

AUG = dict(
    hsv_h=0.015, hsv_s=0.7, hsv_v=0.4,
    degrees=5.0, translate=0.15, scale=0.7, shear=2.0, perspective=0.0005,
    fliplr=0.5, flipud=0.0,
    mosaic=0.8, mixup=0.05, copy_paste=0.0, erasing=0.2,
    close_mosaic=12, multi_scale=0.0,
)

# ===== ĐÁNH GIÁ DETECTOR =====
VAL_CONF = 0.001
VAL_IOU = 0.7
SWEEP_MAX_IMAGES = 500
CONF_GRID = (0.01, 0.02, 0.03, 0.05, 0.08, 0.10, 0.15, 0.20, 0.30, 0.40, 0.50)
IOU_GRID = (0.5, 0.6, 0.7)
MOT_IOU_TH = 0.5

# ===== TRACKING =====
TRACK_SPLITS = ('train', 'val', 'test')
TRACK_CONF = 0.05
TRACK_IOU = 0.5
TRACK_BUFFER = 45
TRACK_ID_IOU = 0.4

# ===== 21 ĐẶC TRƯNG =====
FEATURES = [
    'conf', 'conf_std5', 'conf_delta',
    'track_age', 'hit_ratio', 'time_since_update', 'track_state',
    'min_side_px', 'aspect_dev', 'dist_edge', 'is_trunc',
    'speed_norm', 'pred_uncert',
    'lap_var_log', 'brightness', 'motion_blur_ratio',
    'nearest_dist_norm', 'n_neighbors_r2_norm', 'overlap_max',
    'frame_det_delta', 'n_det_expected_ratio',
]
CONF_WINDOW = 5
PRED_UNCERT_WINDOW = 5
NEIGHBOR_RADIUS_FRAC = 0.10
NEIGHBOR_K = 10
EWMA_ALPHA = 0.3
TRUNC_MARGIN_PX = 2
CROP_RESIZE = 128
FRAME_STRIDE = 1

# ===== CỬA SỔ MẪU =====
WINDOW_SIZE = 60          # 0 = dùng cả track
WINDOW_STRIDE = 1        # dự đoán theo từng frame; cửa sổ chỉ là lịch sử quan sát
MIN_WINDOW_FRAMES = 20

# ===== ẨN DANH =====
ANON_METHOD = 'blur'      # 'blur' | 'pixelate'
ANON_KSIZE = 15
ANON_PIXEL_BLOCK = 8

# ===== RE-ID =====
REID_BACKBONE = 'auto'    # 'auto' | 'osnet_x1_0' | 'resnet50'
REID_INPUT = 256
REID_BATCH = 64
EMBED_STRIDE = 5
PROBE_STRIDE = 10
MAX_PROBE_CROPS = 8
MAX_GALLERY_IDS = 200
GALLERY_MIN_GAP = 30      # giảm khoảng cách để tăng số mẫu có gallery; vẫn loại cửa sổ lân cận
GALLERY_CROSS_SEQUENCE = True
MIN_GALLERY_IDS = 2

# ===== GÁN NHÃN =====
TAU_SELECT_SPLIT = 'val'
LABEL_MIN_MATCH_RATE = 0.5

# ===== MÔ HÌNH RỦI RO =====
XGB_PARAMS = dict(
    n_estimators=400, max_depth=4, learning_rate=0.05,
    subsample=0.9, colsample_bytree=0.9, min_child_weight=2,
    reg_lambda=1.0, objective='binary:logistic', eval_metric='aucpr',
    tree_method='hist', early_stopping_rounds=40, n_jobs=4, random_state=SEED,
)
N_FOLDS = 5
RISK_THRESHOLD = 0.5

# ===== EXPORT =====
EXPORT_ONNX = True
ONNX_OPSET = 17

for _p in (WORK, RUNS_DIR, ARTIFACT, BUILD_DIR):
    _p.mkdir(parents=True, exist_ok=True)

print('CONFIG OK |', MODEL_NAME, '| imgsz', IMGSZ, '| batch', BATCH, '| epochs', EPOCHS)
if USE_SEPARATE_DIRS:
    for _k, _v in (('train', TRAIN_DIR), ('val', VAL_DIR), ('test', TEST_DIR)):
        print(f'data {_k:<5}:', _v, '| exists:', _v.is_dir())
else:
    print('data  :', DATA_ROOT, '| exists:', DATA_ROOT.is_dir())
print('features:', len(FEATURES))
assert len(FEATURES) == 21, 'Phải có đúng 21 đặc trưng'


In [4]:
# ── CELL 3/13 · Chuẩn hoá dataset → YOLO person ────────────
# Vào : DATA_ROOT (hoặc TRAIN_DIR/VAL_DIR/TEST_DIR)
# Ra  : BUILD_DIR/<split>/images+labels, data_person.yaml,
#       sequence_index.csv, gt_boxes.csv, layout.json

import yaml
from PIL import Image
from concurrent.futures import ThreadPoolExecutor

IMG_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}
PSEUDO_MAX_GAP = 5
PSEUDO_IOU = 0.3
SPLITS = ('train', 'val', 'test')


def xyxy_iou(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)), dtype=np.float32)
    a = np.asarray(a, dtype=np.float32)
    b = np.asarray(b, dtype=np.float32)
    x1 = np.maximum(a[:, None, 0], b[None, :, 0])
    y1 = np.maximum(a[:, None, 1], b[None, :, 1])
    x2 = np.minimum(a[:, None, 2], b[None, :, 2])
    y2 = np.minimum(a[:, None, 3], b[None, :, 3])
    inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
    area_a = (a[:, 2] - a[:, 0]) * (a[:, 3] - a[:, 1])
    area_b = (b[:, 2] - b[:, 0]) * (b[:, 3] - b[:, 1])
    union = area_a[:, None] + area_b[None, :] - inter
    return np.where(union > 0, inter / union, 0.0)


def find_layout(path):
    if not path.is_dir():
        return None, None
    queue = [(path, 0)]
    while queue:
        cur, depth = queue.pop(0)
        if (cur / 'images').is_dir() and (cur / 'labels').is_dir():
            return 'yolo', cur
        if (cur / 'annotations').is_dir() and (cur / 'sequences').is_dir():
            return 'raw', cur
        if depth < 2:
            queue += [(p, depth + 1) for p in sorted(cur.iterdir()) if p.is_dir()]
    return None, None


def image_size(path):
    try:
        with Image.open(path) as im:
            return im.size
    except Exception:
        arr = cv2.imread(str(path), cv2.IMREAD_UNCHANGED)
        return (arr.shape[1], arr.shape[0]) if arr is not None else (0, 0)


def parse_annotation(path):
    rows = defaultdict(list)
    for raw in path.read_text(encoding='utf-8', errors='ignore').splitlines():
        f = raw.split(',')
        if len(f) != 10:
            continue
        try:
            frame, tid = int(float(f[0])), int(float(f[1]))
            x, y, w, h = (float(v) for v in f[2:6])
            score, cat = int(float(f[6])), int(float(f[7]))
        except ValueError:
            continue
        if score == 0 or not (1 <= cat <= 11) or w <= 0 or h <= 0:
            continue
        rows[frame].append((cat, x, y, w, h, tid))
    return rows


def copy_many(pairs, workers=16):
    # Kaggle: /kaggle/input là mount FUSE (~1 MB/s/file, overhead mở file rất cao).
    # Symlink vẫn đọc qua mount đó nên phải copy thật về /kaggle/working.
    # Copy song song để bù chi phí mở file.
    todo = [(s, d) for s, d in pairs if not d.exists()]
    if not todo:
        return 0
    with ThreadPoolExecutor(max_workers=workers) as ex:
        list(ex.map(lambda p: shutil.copy2(p[0], p[1]), todo))
    return len(todo)


def write_label(path, lines):
    path.write_text('\n'.join(lines) + ('\n' if lines else ''), encoding='utf-8')


def parse_name(path):
    m = re.match(r'^(.*)_frame(\d+)$', path.stem)
    if m:
        return m.group(1), int(m.group(2))
    m = re.match(r'^(.+?)_(\d+)$', path.stem)
    if m:
        return m.group(1), int(m.group(2))
    return (path.parent.name or '__all__'), 0


def collect_sequences(kind, root):
    if kind == 'raw':
        return sorted(p.name for p in (root / 'sequences').iterdir() if p.is_dir())
    seqs = set()
    for p in (root / 'images').rglob('*'):
        if p.suffix.lower() in IMG_EXTS:
            seqs.add(parse_name(p)[0])
    return sorted(seqs)


def split_sequences(seqs, ratios, seed):
    arr = sorted(seqs)
    random.Random(seed).shuffle(arr)
    n = len(arr)
    a, b, c = ratios
    total = a + b + c
    n_tr = max(1, int(round(n * a / total)))
    n_va = max(1, int(round(n * b / total)))
    if n_tr + n_va >= n:
        n_tr = max(1, n - 2)
        n_va = 1
    return {'train': arr[:n_tr], 'val': arr[n_tr:n_tr + n_va], 'test': arr[n_tr + n_va:]}


def convert_raw(root, split, seq_filter, out, index_rows, gt_rows):
    img_dir, lbl_dir = out / 'images', out / 'labels'
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    ann = {p.stem: parse_annotation(p) for p in (root / 'annotations').glob('*.txt')}
    seqs = [p for p in sorted((root / 'sequences').iterdir()) if p.is_dir()]
    if seq_filter is not None:
        seqs = [p for p in seqs if p.name in seq_filter]

    n_img = 0
    pairs = []
    for seq in seqs:
        files = sorted(p for p in seq.iterdir() if p.suffix.lower() in IMG_EXTS)
        if not files:
            continue
        iw, ih = image_size(files[0])
        rows = ann.get(seq.name, {})
        for src in files:
            frame = int(src.stem) if src.stem.isdigit() else n_img
            flat = f'{seq.name}__{frame:07d}.jpg'
            pairs.append((src, img_dir / flat))
            lines = []
            for cat, x, y, w, h, tid in rows.get(frame, []):
                if cat not in PERSON_CATS:
                    continue
                lines.append(f'0 {(x + w / 2) / iw:.6f} {(y + h / 2) / ih:.6f} {w / iw:.6f} {h / ih:.6f}')
                gt_rows.append({'split': split, 'sequence': seq.name, 'frame': frame,
                                'target_id': int(tid), 'x1': x, 'y1': y,
                                'x2': x + w, 'y2': y + h, 'w': w, 'h': h})
            write_label(lbl_dir / f'{flat[:-4]}.txt', lines)
            index_rows.append({'split': split, 'sequence': seq.name, 'frame': frame,
                               'flat_name': flat, 'source_image': str(src),
                               'width': iw, 'height': ih})
            n_img += 1
    copy_many(pairs)
    return n_img


def convert_yolo(root, split, seq_filter, out, index_rows, gt_rows):
    img_dir, lbl_dir = out / 'images', out / 'labels'
    img_dir.mkdir(parents=True, exist_ok=True)
    lbl_dir.mkdir(parents=True, exist_ok=True)
    sizes = {}
    n_img = 0
    pairs = []
    for src in sorted((root / 'images').rglob('*')):
        if src.suffix.lower() not in IMG_EXTS:
            continue
        seq, frame = parse_name(src)
        if seq_filter is not None and seq not in seq_filter:
            continue
        if seq not in sizes:
            sizes[seq] = image_size(src)
        iw, ih = sizes[seq]

        rel = src.relative_to(root / 'images')
        lab = (root / 'labels' / rel).with_suffix('.txt')
        lines = []
        if lab.is_file():
            for ln in lab.read_text(encoding='utf-8', errors='ignore').splitlines():
                f = ln.split()
                if len(f) != 5:
                    continue
                try:
                    cls = int(float(f[0]))
                    cx, cy, w, h = (float(v) for v in f[1:])
                except ValueError:
                    continue
                if cls not in (0, 1):
                    continue
                x1, y1 = (cx - w / 2) * iw, (cy - h / 2) * ih
                x2, y2 = (cx + w / 2) * iw, (cy + h / 2) * ih
                lines.append(f'0 {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
                gt_rows.append({'split': split, 'sequence': seq, 'frame': frame,
                                'target_id': -1, 'x1': x1, 'y1': y1,
                                'x2': x2, 'y2': y2, 'w': x2 - x1, 'h': y2 - y1})

        flat = f'{seq}__{frame:07d}.jpg'
        pairs.append((src, img_dir / flat))
        write_label(lbl_dir / f'{flat[:-4]}.txt', lines)
        index_rows.append({'split': split, 'sequence': seq, 'frame': frame,
                           'flat_name': flat, 'source_image': str(src),
                           'width': iw, 'height': ih})
        n_img += 1
    copy_many(pairs)
    return n_img


def assign_pseudo_ids(gt_rows):
    groups = defaultdict(list)
    for i, r in enumerate(gt_rows):
        if r['target_id'] < 0:
            groups[(r['split'], r['sequence'])].append(i)

    for key, idxs in groups.items():
        idxs.sort(key=lambda i: gt_rows[i]['frame'])
        by_frame = defaultdict(list)
        for i in idxs:
            by_frame[gt_rows[i]['frame']].append(i)
        active, next_id = {}, 0
        for frame in sorted(by_frame):
            for tid in list(active):
                if frame - active[tid][0] > PSEUDO_MAX_GAP:
                    del active[tid]
            cur = by_frame[frame]
            boxes = np.array([[gt_rows[i][k] for k in ('x1', 'y1', 'x2', 'y2')] for i in cur], float)
            used, used_j = set(), set()
            if active:
                tids = list(active)
                gb = np.array([active[t][1] for t in tids], float)
                iou = xyxy_iou(boxes, gb)
                for flat in np.argsort(-iou, axis=None):
                    a, b = divmod(int(flat), iou.shape[1])
                    if iou[a, b] < PSEUDO_IOU:
                        break
                    if a in used or b in used_j:
                        continue
                    used.add(a)
                    used_j.add(b)
                    gt_rows[cur[a]]['target_id'] = tids[b]
                    active[tids[b]] = (frame, boxes[a])
            for a, i in enumerate(cur):
                if a in used:
                    continue
                gt_rows[i]['target_id'] = next_id
                active[next_id] = (frame, boxes[a])
                next_id += 1
    return sum(1 for r in gt_rows if r['target_id'] < 0)


index_rows, gt_rows = [], []
layout_info = {'mode': 'separate' if USE_SEPARATE_DIRS else 'auto_split', 'sources': {}}

_free_gb = shutil.disk_usage(WORK).free / 2 ** 30
print(f'đĩa trống /kaggle/working: {_free_gb:.1f} GB (cần ~12 GB để copy ảnh)')
if _free_gb < 12:
    print('CẢNH BÁO: thiếu dung lượng. Dọn /kaggle/working hoặc giảm dữ liệu.')

if USE_SEPARATE_DIRS:
    dirs = {'train': TRAIN_DIR, 'val': VAL_DIR, 'test': TEST_DIR}
    for split in SPLITS:
        kind, root = find_layout(dirs[split])
        if kind is None:
            raise FileNotFoundError(f'{split}: không thấy dữ liệu trong {dirs[split]}')
        layout_info['sources'][split] = {'kind': kind, 'root': str(root)}
        out = BUILD_DIR / split
        if kind == 'raw':
            n = convert_raw(root, split, None, out, index_rows, gt_rows)
        else:
            n = convert_yolo(root, split, None, out, index_rows, gt_rows)
        print(f'{split:<5} {n:>5} ảnh ({kind})')
else:
    src = DATA_ROOT
    kind, root = find_layout(src)
    if kind is None:
        raise FileNotFoundError(
            f'Không thấy annotations/+sequences/ hoặc images/+labels/ trong {src}. '
            f'Kiểm tra lại DATA_ROOT hoặc bật USE_SEPARATE_DIRS.'
        )
    all_seqs = collect_sequences(kind, root)
    if len(all_seqs) < 3:
        raise ValueError(f'Chỉ có {len(all_seqs)} sequence, cần >=3 để tự chia. '
                         f'Đặt USE_SEPARATE_DIRS = True nếu đã có 3 thư mục riêng.')
    seqs = split_sequences(all_seqs, SPLIT_RATIOS, SPLIT_SEED)
    layout_info.update({'kind': kind, 'root': str(root), 'sequences': seqs})
    print('auto split:', {k: len(v) for k, v in seqs.items()})
    for split in SPLITS:
        out = BUILD_DIR / split
        if kind == 'raw':
            n = convert_raw(root, split, set(seqs[split]), out, index_rows, gt_rows)
        else:
            n = convert_yolo(root, split, set(seqs[split]), out, index_rows, gt_rows)
        print(f'{split:<5} {n:>5} ảnh | {seqs[split]}')

n_pseudo = sum(1 for r in gt_rows if r['target_id'] < 0)
assign_pseudo_ids(gt_rows)
layout_info['pseudo_identity'] = n_pseudo > 0
layout_info['pseudo_rows'] = n_pseudo

df_idx = pd.DataFrame(index_rows).sort_values(['split', 'sequence', 'frame'])
df_gt = pd.DataFrame(gt_rows).sort_values(['split', 'sequence', 'frame', 'target_id'])
df_idx.to_csv(SEQ_INDEX, index=False)
df_gt.to_csv(GT_CSV, index=False)
LAYOUT_JSON.write_text(json.dumps(layout_info, indent=2, ensure_ascii=False), encoding='utf-8')

DATA_YAML.write_text(yaml.safe_dump({
    'path': str(WORK),
    'train': str(BUILD_DIR / 'train' / 'images'),
    'val': str(BUILD_DIR / 'val' / 'images'),
    'test': str(BUILD_DIR / 'test' / 'images'),
    'nc': 1,
    'names': {0: 'person'},
}, sort_keys=False, allow_unicode=True), encoding='utf-8')

print('\nsequence_index.csv:', len(df_idx), 'dòng')
print('gt_boxes.csv      :', len(df_gt), 'box |', df_gt.target_id.nunique(), 'identity')
if n_pseudo:
    print(f'CẢNH BÁO: {n_pseudo} box không có target_id → dùng pseudo-ID (xấp xỉ).')
print(DATA_YAML.read_text(encoding='utf-8'))


đĩa trống /kaggle/working: 19.5 GB (cần ~12 GB để copy ảnh)
train 24201 ảnh (raw)
val    2846 ảnh (raw)
test   6635 ảnh (raw)

sequence_index.csv: 33682 dòng
gt_boxes.csv      : 519196 box | 453 identity
path: /kaggle/working
train: /kaggle/working/privacy_build/train/images
val: /kaggle/working/privacy_build/val/images
test: /kaggle/working/privacy_build/test/images
nc: 1
names:
  0: person



In [ ]:
# ── CELL 4/13 · Audit dataset + thong ke identity ──────────
# Vao : BUILD_DIR/<split>, sequence_index.csv, gt_boxes.csv
# Ra  : artifacts/dataset_audit.json, artifacts/identity_stats.json


df_idx = add_validation_split(pd.read_csv(SEQ_INDEX), seed=SEED, validation_fraction=0.20)
df_idx.to_csv(SEQ_INDEX, index=False)
if GT_CSV.is_file():
    _gt = pd.read_csv(GT_CSV)
    _seq_split = df_idx[['sequence', 'split']].drop_duplicates()
    _gt = _gt.drop(columns=['split'], errors='ignore').merge(_seq_split, on='sequence', how='inner')
    _gt.to_csv(GT_CSV, index=False)
print('tracking split:', df_idx.groupby('split').sequence.nunique().to_dict())
df_gt = pd.read_csv(GT_CSV)
audit, id_stats = [], {}

for split in SPLITS:
    sub_idx = df_idx[df_idx.split == split]
    sub_gt = df_gt[df_gt.split == split]
    images = sorted((BUILD_DIR / split / 'images').glob('*.jpg'))
    counts = Counter()
    empty = 0
    for img in images:
        lab = BUILD_DIR / split / 'labels' / f'{img.stem}.txt'
        rows = [ln for ln in lab.read_text(encoding='utf-8', errors='ignore').splitlines() if ln.strip()] if lab.is_file() else []
        if not rows:
            empty += 1
        counts[0] += len(rows)

    per_identity = sub_gt.groupby(['sequence', 'target_id']).size()
    lens = sorted(int(v) for v in per_identity.values)
    info = {
        'split': split,
        'sequences': int(sub_idx.sequence.nunique()),
        'images': len(images),
        'frames': int(sub_idx.frame.nunique()),
        'boxes_person': int(counts[0]),
        'boxes_per_frame': round(counts[0] / max(len(images), 1), 2),
        'empty_labels': empty,
        'identities': int(sub_gt.groupby(['sequence', 'target_id']).ngroups),
        'identity_tracks': int(len(per_identity)),
        'track_len_min': lens[0] if lens else 0,
        'track_len_median': int(np.median(lens)) if lens else 0,
        'track_len_p90': lens[int(0.9 * (len(lens) - 1))] if lens else 0,
        'track_len_max': lens[-1] if lens else 0,
        'tracks_ge_30': int(sum(1 for v in lens if v >= 30)),
        'tracks_ge_90': int(sum(1 for v in lens if v >= 90)),
    }
    audit.append(info)
    id_stats[split] = {
        'identities': int(sub_gt.groupby(['sequence', 'target_id']).ngroups),
        'sequence_ids': sub_gt.groupby('sequence').target_id.nunique().to_dict(),
    }

    print('=' * 78)
    for k, v in info.items():
        print(f'  {k:<20}: {v}')

(ARTIFACT / 'dataset_audit.json').write_text(json.dumps(audit, indent=2, ensure_ascii=False), encoding='utf-8')
(ARTIFACT / 'identity_stats.json').write_text(json.dumps(id_stats, indent=2, ensure_ascii=False), encoding='utf-8')

for a in audit:
    assert a['images'] > 0, f"Split {a['split']} khong co anh"
print('
Audit xong. Tong identity:', int(df_gt.target_id.nunique()))


In [ ]:
# ── CELL 5/13 · Train YOLO26n person ──────────────────────
# Vào : DATA_YAML
# Ra  : RUNS_DIR/<RUN_NAME>/weights/yolo26s.pt

def load_base_model():
    for name in [MODEL_NAME] + FALLBACK:
        try:
            m = YOLO(name)
            print('base model:', name)
            return m
        except Exception as e:
            print('không load được', name, '->', str(e)[:140])
    raise RuntimeError('Không tải được checkpoint nào')


def is_oom(err):
    s = str(err).lower()
    return 'out of memory' in s or ('cuda' in s and 'memory' in s)


_ladder = [(IMGSZ, BATCH), (1024, 12), (1024, 8), (896, 12), (768, 16)]
_seen = set()
LADDER = [x for x in _ladder if not (x in _seen or _seen.add(x))]

train_kwargs = dict(
    data=str(DATA_YAML), project=str(RUNS_DIR), name=RUN_NAME, exist_ok=True,
    pretrained=True, epochs=EPOCHS, workers=WORKERS, cache=False, amp=True,
    seed=SEED, deterministic=True, patience=PATIENCE, cos_lr=True,
    optimizer='AdamW', lr0=0.002, lrf=0.01, weight_decay=0.0005, warmup_epochs=3.0,
    box=7.5, cls=0.5, dfl=1.5, max_det=MAX_DET,
    plots=True, save=True, save_period=5, val=True, verbose=True,
    **AUG,
)

IMGSZ_USED, BATCH_USED = IMGSZ, BATCH
trained = False
t0 = time.perf_counter()

for imgsz_try, batch_try in LADDER:
    for dev_try in (DEVICE, str(DEVICE_EVAL)):
        try:
            print(f'>>> train | imgsz={imgsz_try} batch={batch_try} device={dev_try}', flush=True)
            _model = load_base_model()
            _model.train(imgsz=imgsz_try, batch=batch_try, device=dev_try, **train_kwargs)
            IMGSZ_USED, BATCH_USED = imgsz_try, batch_try
            trained = True
            del _model
            gc.collect()
            torch.cuda.empty_cache()
            break
        except Exception as e:
            print('FAIL:', str(e)[:400])
            gc.collect()
            torch.cuda.empty_cache()
            if is_oom(e) or dev_try == DEVICE:
                continue
            raise
    if trained:
        break

assert trained, 'Train thất bại ở mọi cấu hình.'
print(f'Train xong {(time.perf_counter() - t0) / 60:.1f} phút | imgsz={IMGSZ_USED} batch={BATCH_USED}')

def _find_trained_ckpt(runs_dir, run_name):
    _cands = [
        runs_dir / run_name / 'weights' / 'best.pt',
        runs_dir / run_name / 'weights' / 'last.pt',
    ]
    _cands += sorted(runs_dir.glob('**/weights/best.pt'), key=lambda _p: _p.stat().st_mtime, reverse=True)
    _cands += sorted(runs_dir.glob('**/weights/last.pt'), key=lambda _p: _p.stat().st_mtime, reverse=True)
    for _c in dict.fromkeys(_cands):
        if _c.is_file():
            return _c
    return None


_src_ckpt = _find_trained_ckpt(RUNS_DIR, RUN_NAME)
if _src_ckpt is None:
    raise RuntimeError(f'Không thấy checkpoint huấn luyện trong {RUNS_DIR}.')
BEST_MODEL = RUNS_DIR / RUN_NAME / 'weights' / 'yolo26s.pt'
if not BEST_MODEL.is_file() or BEST_MODEL.stat().st_mtime < _src_ckpt.stat().st_mtime:
    BEST_MODEL.parent.mkdir(parents=True, exist_ok=True)
    shutil.copy2(_src_ckpt, BEST_MODEL)
assert BEST_MODEL.is_file(), f'Không thấy yolo26s.pt tại {BEST_MODEL}'
print('yolo26s.pt:', BEST_MODEL, '|', f'{BEST_MODEL.stat().st_size / 2**20:.1f} MiB')

# ── Don RAM/VRAM sau khi train ──────────────────────────────
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
print('Da don RAM/VRAM sau Cell 5 (train).')


In [ ]:
# ── CELL PRE-6 · Lưu toàn bộ cấu hình phục vụ notebook4266 ─────
# Lưu tất cả biến cấu hình, đường dẫn và siêu tham số từ notebook4234
# để notebook4266 có thể import và chạy tiếp mà không cần chạy lại các cell trước.

import json
from pathlib import Path

_CONFIG = {}

# Danh sách các biến cấu hình cần phục hồi cho notebook4266
_config_names = [
    'WORK', 'BUILD_DIR', 'RUNS_DIR', 'ARTIFACT',
    'DATA_ROOT', 'USE_SEPARATE_DIRS', 'TRAIN_DIR', 'VAL_DIR', 'TEST_DIR',
    'SPLIT_RATIOS', 'SPLIT_SEED',
    'SEQ_INDEX', 'GT_CSV', 'DATA_YAML', 'LAYOUT_JSON',
    'TRACKS_CSV', 'FEATURES_CSV', 'SAMPLES_CSV', 'ATTACK_CSV', 'TAU_JSON',
    'RISK_CSV', 'SCHEMA_JSON', 'RISK_MODEL_JSON', 'RISK_OOF_CSV', 'RISK_METRICS_JSON',
    'DET_METRICS_JSON',
    'MODEL_NAME', 'FALLBACK', 'PERSON_CATS', 'IMGSZ', 'BATCH', 'EPOCHS',
    'WORKERS', 'DEVICE', 'DEVICE_EVAL', 'SEED', 'PATIENCE', 'RUN_NAME', 'MAX_DET',
    'AUG', 'VAL_CONF', 'VAL_IOU', 'SWEEP_MAX_IMAGES', 'CONF_GRID', 'IOU_GRID', 'MOT_IOU_TH',
    'TRACK_SPLITS', 'TRACK_CONF', 'TRACK_IOU', 'TRACK_BUFFER', 'TRACK_ID_IOU',
    'FEATURES', 'CONF_WINDOW', 'PRED_UNCERT_WINDOW', 'NEIGHBOR_RADIUS_FRAC',
    'NEIGHBOR_K', 'EWMA_ALPHA', 'TRUNC_MARGIN_PX', 'CROP_RESIZE', 'FRAME_STRIDE',
    'WINDOW_SIZE', 'WINDOW_STRIDE', 'MIN_WINDOW_FRAMES',
    'ANON_METHOD', 'ANON_KSIZE', 'ANON_PIXEL_BLOCK',
    'REID_BACKBONE', 'REID_INPUT', 'REID_BATCH',
    'XGB_PARAMS', 'N_FOLDS', 'RISK_THRESHOLD',
    'EXPORT_ONNX', 'ONNX_OPSET',
    'BEST_MODEL', 'IMGSZ_USED', 'BATCH_USED',
]

for _name in _config_names:
    if _name in globals():
        _val = globals()[_name]
        # Chuyển Path thành chuỗi để serialize JSON
        if isinstance(_val, Path):
            _val = str(_val)
        elif isinstance(_val, dict):
            _val = {str(k): str(v) if isinstance(v, Path) else v for k, v in _val.items()}
        elif isinstance(_val, (tuple, list)):
            _val = [_v if not isinstance(_v, Path) else str(_v) for _v in _val]
        _CONFIG[_name] = _val

_CONFIG_PATH = ARTIFACT / 'notebook4234_config.json'
_CONFIG_PATH.parent.mkdir(parents=True, exist_ok=True)
_CONFIG_PATH.write_text(
    json.dumps(_CONFIG, indent=2, ensure_ascii=False, default=str), encoding='utf-8'
)

print('Đã lưu cấu hình phục vụ notebook4266:', _CONFIG_PATH)
print('Số biến cấu hình:', len(_CONFIG))


In [ ]:
# Cell 6 can be run after a kernel restart or without rerunning training.
from pathlib import Path
# Recover the trained checkpoint and evaluation defaults explicitly.
if 'IMGSZ_USED' not in globals():
    IMGSZ_USED = int(globals().get('IMGSZ', 1280))
if 'BATCH_USED' not in globals():
    BATCH_USED = int(globals().get('BATCH', 1))
if 'BEST_MODEL' not in globals() or not Path(BEST_MODEL).is_file():
    _runs_dir = Path(globals().get('RUNS_DIR', Path('/kaggle/working/runs')))
    _run_name = str(globals().get('RUN_NAME', 'yolo26s_visdrone_recall_t4x2'))
    # Ưu tiên file chuẩn hóa từ Cell 5, sau đó quét mọi checkpoint đã train (best.pt/last.pt/*.pt).
    _candidates = []
    _expected = _runs_dir / _run_name / 'weights' / 'yolo26s.pt'
    if _expected.is_file():
        _candidates.append(_expected)
    _candidates += sorted(_runs_dir.glob('**/weights/best.pt'), key=lambda _p: _p.stat().st_mtime, reverse=True)
    _candidates += sorted(_runs_dir.glob('**/weights/last.pt'), key=lambda _p: _p.stat().st_mtime, reverse=True)
    _candidates += sorted(_runs_dir.glob('**/weights/*.pt'), key=lambda _p: _p.stat().st_mtime, reverse=True)
    _seen = set()
    _candidates = [c for c in _candidates if not (c in _seen or _seen.add(c))]
    if _candidates:
        BEST_MODEL = _candidates[0]
        print('Cell 6 dùng checkpoint:', BEST_MODEL)
    else:
        # Không có checkpoint đã train: dùng base model để các cell sau vẫn chạy (cảnh báo).
        print('CẢNH BÁO: Không tìm thấy checkpoint đã train, dùng base model thay thế.')
        _bm = None
        for _name in [MODEL_NAME] + FALLBACK:
            try:
                _bm = YOLO(_name)
                print('base model:', _name)
                break
            except Exception as _e:
                print('không load được', _name, '->', str(_e)[:120])
        if _bm is None:
            raise FileNotFoundError(
                f'Không tìm thấy yolo26s.pt (hoặc checkpoint nào) trong {_runs_dir}. '
                'Hãy chạy Cell 5 train trước.'
            )
        BEST_MODEL = Path(_name)
        print('Cell 6 dùng base model:', BEST_MODEL)
if 'DATA_YAML' not in globals():
    DATA_YAML = Path('/kaggle/working/data_person.yaml')
if 'GT_CSV' not in globals():
    GT_CSV = Path('/kaggle/working/gt_boxes.csv')
if 'ARTIFACT' not in globals():
    ARTIFACT = Path('/kaggle/working/artifacts')
if 'DET_METRICS_JSON' not in globals():
    DET_METRICS_JSON = ARTIFACT / 'det_metrics.json'
ARTIFACT.mkdir(parents=True, exist_ok=True)
print('Cell 6 checkpoint:', BEST_MODEL)

# ── CELL 6/13 · Eval detector + sweep conf/iou ─────────────
# Vào : yolo26s.pt, DATA_YAML, GT_CSV
# Ra  : det_metrics.json, threshold_sweep.csv, CONF_BEST, IOU_BEST,
#       cell6_checkpoint/eval_checkpoint.json (metadata cho Cell 7 import),
#       cell6_checkpoint/detection_results.npz (boxes/confs/frame_keys val để Cell 7 dùng tiếp)
#
# CẤU HÌNH NGƯỠNG RAM TỐI ĐA:
#   RAM_LIMIT_GB   : ngưỡng RAM tối đa (GiB). None = tự dò (cgroup/Kaggle; nếu không được thì tổng RAM).
#   RAM_STOP_FRAC  : tạm ngừng khi dùng >= RAM_STOP_FRAC × ngưỡng (mặc định 0.85 = 85%).
# Khi RAM >= RAM_LIMIT_GB × RAM_STOP_FRAC:
#   • lưu kết quả đã chạy (det_metrics.json, threshold_sweep.csv, eval_checkpoint.json) vào output
#   • ném ngoại lệ RamThresholdReached để tạm ngừng tiến trình (chạy lại cell này để tiếp tục).

import os as _os
_os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'

try:
    import psutil
    _HAS_PSUTIL = True
except Exception:
    _HAS_PSUTIL = False

# ---------- Cấu hình ngưỡng RAM ----------
RAM_LIMIT_GB = None        # None = tự dò; ví dụ đặt 25.0 để ép ngưỡng 25 GiB
RAM_STOP_FRAC = 0.85       # tạm ngừng khi đạt >= 85% ngưỡng RAM


class RamThresholdReached(Exception):
    """Ném khi RAM chạm ngưỡng tối đa → tạm ngừng tiến trình."""


def _read_cgroup_int(path):
    try:
        return int(Path(path).read_text().strip())
    except Exception:
        return None


def _cgroup_used_gib():
    cur = _read_cgroup_int('/sys/fs/cgroup/memory.current') or _read_cgroup_int(
        '/sys/fs/cgroup/memory/memory.usage_in_bytes')
    if cur is None:
        return None
    inactive = 0
    for stat in ('/sys/fs/cgroup/memory.stat', '/sys/fs/cgroup/memory/memory.stat'):
        try:
            for line in Path(stat).read_text().splitlines():
                key, _, val = line.partition(' ')
                if key in ('inactive_file', 'total_inactive_file'):
                    inactive = int(val)
                    break
            break
        except Exception:
            continue
    return max(cur - inactive, 0) / 2**30


def _cgroup_limit_gib():
    for path in ('/sys/fs/cgroup/memory.max', '/sys/fs/cgroup/memory/memory.limit_in_bytes'):
        v = _read_cgroup_int(path)
        if v is not None and v < 2**60:
            return v / 2**30
    return None


def _ram_used_gib():
    used = _cgroup_used_gib()
    if used is not None:
        return used
    if _HAS_PSUTIL:
        try:
            return psutil.Process().memory_info().rss / 2**30
        except Exception:
            return None
    return None


def _ram_limit_gib():
    if RAM_LIMIT_GB:
        return float(RAM_LIMIT_GB)
    lim = _cgroup_limit_gib()
    if lim:
        return lim
    if _HAS_PSUTIL:
        try:
            return psutil.virtual_memory().total / 2**30
        except Exception:
            return None
    return None


def _guard_ram(stage_name=""):
    """Kiểm tra RAM. Nếu >= ngưỡng tối đa thì lưu output và tạm ngừng tiến trình (raise)."""
    limit = _ram_limit_gib()
    if limit is None:
        return
    stop_gib = limit * RAM_STOP_FRAC
    used = _ram_used_gib()
    if used is None:
        return
    if used >= stop_gib:
        # Lưu kết quả đã chạy vào output trước khi tạm ngừng
        try:
            save_intermediate_results()
        except Exception as _e:
            print(f'[RAM guard] lưu output thất bại: {_e}', flush=True)
        print(f'[RAM guard] Đã chạm ngưỡng {stop_gib:.1f}G/{limit:.1f}G '
              f'(dùng {used:.1f}G) tại {stage_name} → đã lưu output, tạm ngừng tiến trình.', flush=True)
        raise RamThresholdReached(
            f'RAM {used:.1f}G ≥ ngưỡng {stop_gib:.1f}G tại {stage_name}. '
            'Đã lưu kết quả vào output. Chạy lại cell này để tiếp tục.')


# Lưu trạng thái run để notebook4266 biết tiếp tục từ đâu
_RUN_STATE = {
    'stage': 'start',
    'det_metrics': {},
    'threshold_sweep_rows': [],
    'best': {'conf': TRACK_CONF, 'iou': TRACK_IOU, 'f1': -1.0, 'recall': -1.0},
    'eval_imgsz_used': None,
    'ram_stopped': False,
}


def save_intermediate_results():
    """Lưu tất cả kết quả trung gian đã có để notebook4266 import và chạy tiếp."""
    PRE_EVAL_DIR.mkdir(parents=True, exist_ok=True)
    # Lưu det_metrics đã có
    if 'det_metrics' in globals() and len(det_metrics) > 0:
        DET_METRICS_JSON.write_text(
            json.dumps(det_metrics, indent=2, ensure_ascii=False), encoding='utf-8'
        )
    # Lưu threshold_sweep đã tính
    if len(_RUN_STATE['threshold_sweep_rows']) > 0:
        import pandas as _pd
        _pd.DataFrame(_RUN_STATE['threshold_sweep_rows']).to_csv(
            ARTIFACT / 'threshold_sweep.csv', index=False
        )
    # Lưu eval_checkpoint metadata
    _eval_checkpoint = {
        'CONF_BEST': float(_RUN_STATE['best']['conf']),
        'IOU_BEST': float(_RUN_STATE['best']['iou']),
        'IMGSZ_USED': int(_RUN_STATE['eval_imgsz_used']) if _RUN_STATE['eval_imgsz_used'] else int(IMGSZ_USED),
        'EVAL_DEVICE': EVAL_DEVICE,
        'EVAL_HALF': EVAL_HALF,
        'TRACK_CONF': float(TRACK_CONF),
        'TRACK_IOU': float(TRACK_IOU),
        'MAX_DET': int(MAX_DET),
        'VAL_CONF': float(VAL_CONF),
        'VAL_IOU': float(VAL_IOU),
        'det_metrics': _RUN_STATE['det_metrics'],
        'val_frame_keys': _RUN_STATE.get('val_frame_keys', []),
        'val_image_paths': _RUN_STATE.get('val_image_paths', []),
        'det_metrics_path': str(DET_METRICS_JSON),
        'threshold_sweep_path': str(ARTIFACT / 'threshold_sweep.csv'),
        'ram_stopped': True,
        'stage': _RUN_STATE['stage'],
    }
    (PRE_EVAL_DIR / 'eval_checkpoint.json').write_text(
        json.dumps(_eval_checkpoint, indent=2, ensure_ascii=False, default=str), encoding='utf-8'
    )
    print('Đã lưu kết quả trung gian do vượt ngưỡng RAM:', PRE_EVAL_DIR / 'eval_checkpoint.json')


# ---- Lưu trạng thái cũ và dọn RAM/VRAM trước khi chạy Cell 6 ----
PRE_EVAL_DIR = ARTIFACT / 'cell6_checkpoint'
PRE_EVAL_DIR.mkdir(parents=True, exist_ok=True)
PRE_EVAL_STATE = {
    'det_metrics_available': bool('det_metrics' in globals()),
    'threshold_sweep_available': bool('df_sweep' in globals()),
    'conf_best': globals().get('CONF_BEST'),
    'iou_best': globals().get('IOU_BEST'),
    'imgsz_used': globals().get('IMGSZ_USED'),
    'batch_used': globals().get('BATCH_USED'),
}
if 'det_metrics' in globals():
    (PRE_EVAL_DIR / 'det_metrics.previous.json').write_text(
        json.dumps(det_metrics, indent=2, ensure_ascii=False), encoding='utf-8'
    )
if 'df_sweep' in globals() and isinstance(df_sweep, pd.DataFrame):
    df_sweep.to_csv(PRE_EVAL_DIR / 'threshold_sweep.previous.csv', index=False)
(PRE_EVAL_DIR / 'pre_eval_state.json').write_text(
    json.dumps(PRE_EVAL_STATE, indent=2, ensure_ascii=False, default=str), encoding='utf-8'
)
print('Cell 6 pre-run state saved:', PRE_EVAL_DIR)
print(json.dumps(PRE_EVAL_STATE, indent=2, ensure_ascii=False, default=str))
for _name in ('eval_model', 'sweep_model', 'model', 'm', 'results', 'val_results',
              'det_results', 'dets', 'df_sweep', 'df_idx', 'df_gt', 'val_idx',
              'val_gt', 'gt_map', 'paths', 'frame_keys', 'rows', 'counts', 'best'):
    globals().pop(_name, None)
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
print('Cell 6 RAM/VRAM cleanup complete; starting detector evaluation.')

# Kiểm tra RAM ban đầu
_guard_ram('trước khi bắt đầu')

EVAL_BATCH_LADDER = tuple(dict.fromkeys((1,)))
# ƯU TIÊN RECALL: giữ imgsz cao (1024, 768) để thấy đối tượng nhỏ trên VisDrone.
# Chỉ giảm xuống 640 nếu OOM, không đi xuống 480/320 vì làm giảm recall nghiêm trọng.
_EVAL_IMGSZ_CANDIDATES = [int(IMGSZ_USED) if int(IMGSZ_USED) <= 1024 else 1024, 1024, 768, 640]
_EVAL_IMGSZ_CANDIDATES = [x for x in _EVAL_IMGSZ_CANDIDATES if x > 0]
EVAL_IMGSZ_LADDER = tuple(dict.fromkeys(_EVAL_IMGSZ_CANDIDATES))

EVAL_DEVICE = DEVICE_EVAL if torch.cuda.is_available() else 'cpu'
EVAL_HALF = bool(torch.cuda.is_available())
eval_model = YOLO(str(BEST_MODEL))

def eval_split_with_oom_retry(split_name):
    last_error = None
    global eval_model, EVAL_DEVICE, EVAL_HALF
    for eval_imgsz in EVAL_IMGSZ_LADDER:
        for eval_batch in EVAL_BATCH_LADDER:
            # Kiểm tra RAM trước mỗi lần eval
            _guard_ram(f'eval {split_name} imgsz={eval_imgsz}')
            try:
                print(f'eval {split_name}: imgsz={eval_imgsz}, batch={eval_batch}', flush=True)
                result = eval_model.val(
                    data=str(DATA_YAML), split=split_name, imgsz=eval_imgsz,
                    batch=eval_batch, conf=VAL_CONF, iou=VAL_IOU, max_det=MAX_DET,
                    device=EVAL_DEVICE, workers=0, cache=False, plots=False,
                    half=EVAL_HALF, verbose=False,
                )
                return result, eval_imgsz, eval_batch
            except Exception as exc:
                last_error = exc
                if not is_oom(exc):
                    raise
                print(f'WARNING: eval OOM at imgsz={eval_imgsz}, batch={eval_batch}; retrying smaller', flush=True)
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                del eval_model
                gc.collect()
                if torch.cuda.is_available():
                    torch.cuda.empty_cache()
                    torch.cuda.ipc_collect()
                eval_model = YOLO(str(BEST_MODEL))
    if last_error is not None and torch.cuda.is_available():
        print(f'GPU OOM for {split_name}; retrying on CPU at imgsz=768.', flush=True)
        del eval_model
        gc.collect()
        torch.cuda.empty_cache()
        torch.cuda.ipc_collect()
        EVAL_DEVICE = 'cpu'
        EVAL_HALF = False
        eval_model = YOLO(str(BEST_MODEL))
        return eval_model.val(
            data=str(DATA_YAML), split=split_name, imgsz=768,
            batch=1, conf=VAL_CONF, iou=VAL_IOU, max_det=MAX_DET,
            device='cpu', workers=0, cache=False, plots=False,
            half=False, verbose=False,
        ), 768, 1
    raise RuntimeError('Cell 6 evaluation exhausted OOM ladder') from last_error

det_metrics = {}
_eval_imgsz_used = None
ram_stopped_early = False

for split in ('val', 'test'):
    if ram_stopped_early:
        break
    try:
        m, _eval_imgsz_used, _eval_batch_used = eval_split_with_oom_retry(split)
    except RamThresholdReached:
        ram_stopped_early = True
        _RUN_STATE['stage'] = f'eval_{split}'
        _RUN_STATE['det_metrics'] = det_metrics
        break
    if m is None:
        # RAM vượt ngưỡng (trường hợp _guard_ram không raise)
        ram_stopped_early = True
        _RUN_STATE['stage'] = f'eval_{split}'
        _RUN_STATE['det_metrics'] = det_metrics
        break
    det_metrics[split] = {
        'mAP50': float(m.box.map50),
        'mAP50_95': float(m.box.map),
        'precision': float(m.box.mp),
        'recall': float(m.box.mr),
        'f1': float(m.box.f1.mean()),
    }
    DET_METRICS_JSON.write_text(json.dumps(det_metrics, indent=2, ensure_ascii=False), encoding='utf-8')
    print(split.upper(), {k: round(v, 4) for k, v in det_metrics[split].items()})
    _RUN_STATE['det_metrics'] = det_metrics
    _RUN_STATE['eval_imgsz_used'] = _eval_imgsz_used
    del m
    gc.collect()
    torch.cuda.empty_cache()

# ---- sweep conf/iou trên val ---- (ưu tiên recall cao)
df_idx = pd.read_csv(SEQ_INDEX)
df_gt = pd.read_csv(GT_CSV)
val_idx = df_idx[df_idx.split == 'val'].sort_values(['sequence', 'frame'])
val_idx = val_idx.iloc[:SWEEP_MAX_IMAGES]
val_gt = df_gt[df_gt.split == 'val']
gt_map = defaultdict(list)
for r in val_gt.itertuples():
    gt_map[(r.sequence, int(r.frame))].append([r.x1, r.y1, r.x2, r.y2])

paths = [BUILD_DIR / 'val' / 'images' / n for n in val_idx.flat_name]
frame_keys = [(s, int(f)) for s, f in zip(val_idx.sequence, val_idx.frame)]

rows = []
RECALL_PRECISION_FLOOR = 0.25  # giảm ngưỡng để ưu tiên recall cao hơn
best = {'recall': -1.0, 'f1': -1.0, 'conf': TRACK_CONF, 'iou': TRACK_IOU}

SWEEP_CHECKPOINT = ARTIFACT / 'cell6_checkpoint' / 'threshold_sweep.partial.csv'
SWEEP_CHECKPOINT.parent.mkdir(parents=True, exist_ok=True)
if SWEEP_CHECKPOINT.is_file():
    SWEEP_CHECKPOINT.unlink()

if not ram_stopped_early:
    try:
        for iou_nms in IOU_GRID:
            # Kiểm tra RAM trước mỗi vòng iou
            _guard_ram(f'sweep iou={iou_nms}')

            counts = {float(conf_th): [0, 0, 0] for conf_th in CONF_GRID}
            _sweep_frame_pos = 0
            # Predict with OOM retry: giữ imgsz cao để recall tốt, chỉ giảm nếu OOM
            _pred_ok = False
            for _try_imgsz in [int(IMGSZ_USED) if int(IMGSZ_USED) <= 1024 else 1024, 768, 640]:
                try:
                    all_boxes = []
                    all_confs = []
                    for res in eval_model.predict(
                        source=[str(p) for p in paths], conf=0.01, iou=iou_nms,
                        imgsz=_try_imgsz, max_det=MAX_DET, device=EVAL_DEVICE,
                        batch=1, half=EVAL_HALF, stream=True, verbose=False,
                    ):
                        b = res.boxes
                        d = b.xyxy.cpu().numpy() if b is not None and len(b) else np.zeros((0, 4))
                        if len(d):
                            d = np.concatenate([d, b.conf.cpu().numpy()[:, None]], 1)
                        # stream output follows source order; keep an explicit counter to avoid retaining detections.
                        seq, frame = frame_keys[_sweep_frame_pos]
                        _sweep_frame_pos += 1
                        g = np.array(gt_map.get((seq, frame), []), float).reshape(-1, 4)
                        for conf_th in CONF_GRID:
                            dd = d[d[:, 4] >= conf_th][:, :4] if len(d) else np.zeros((0, 4))
                            if len(g) == 0:
                                counts[float(conf_th)][1] += len(dd)
                                continue
                            if len(dd) == 0:
                                counts[float(conf_th)][2] += len(g)
                                continue
                            iou = xyxy_iou(g, dd)
                            used_g, used_d = set(), set()
                            for flat in np.argsort(-iou, axis=None):
                                a, bidx = divmod(int(flat), iou.shape[1])
                                if iou[a, bidx] < MOT_IOU_TH:
                                    break
                                if a in used_g or bidx in used_d:
                                    continue
                                used_g.add(a); used_d.add(bidx)
                            counts[float(conf_th)][0] += len(used_g)
                            counts[float(conf_th)][2] += len(g) - len(used_g)
                            counts[float(conf_th)][1] += len(dd) - len(used_d)
                    _pred_ok = True
                    _eval_imgsz_used = _try_imgsz
                    break
                except Exception as _exc:
                    if not is_oom(_exc):
                        raise
                    print(f'WARNING: predict OOM at imgsz={_try_imgsz}; retrying smaller', flush=True)
                    gc.collect()
                    if torch.cuda.is_available():
                        torch.cuda.empty_cache()
            if not _pred_ok:
                raise RuntimeError('Predict exhausted OOM ladder')

            for conf_th in CONF_GRID:
                tp, fp, fn = counts[float(conf_th)]
                p = tp / max(tp + fp, 1); r = tp / max(tp + fn, 1)
                f1 = 2 * p * r / max(p + r, 1e-9)
                rows.append({'conf': conf_th, 'iou': iou_nms, 'precision': p, 'recall': r,
                             'f1': f1, 'tp': tp, 'fp': fp, 'fn': fn})
                # Ưu tiên recall cao: chọn best dựa trên recall trước, f1 sau
                candidate = (r >= RECALL_PRECISION_FLOOR, r, f1)
                current = (best.get('recall', -1.0) >= RECALL_PRECISION_FLOOR, best.get('recall', -1.0), best.get('f1', -1.0))
                if candidate > current:
                    best = {'f1': f1, 'conf': conf_th, 'iou': iou_nms, 'precision': p, 'recall': r}
            _RUN_STATE['threshold_sweep_rows'] = rows
            _RUN_STATE['best'] = best
            _RUN_STATE['eval_imgsz_used'] = _eval_imgsz_used
            pd.DataFrame(rows).to_csv(SWEEP_CHECKPOINT, index=False)
            pd.DataFrame(rows).to_csv(ARTIFACT / 'threshold_sweep.csv', index=False)
            del counts
            gc.collect()
            if torch.cuda.is_available(): torch.cuda.empty_cache()
    except RamThresholdReached:
        ram_stopped_early = True
        _RUN_STATE['stage'] = 'sweep'
        _RUN_STATE['det_metrics'] = det_metrics
        _RUN_STATE['threshold_sweep_rows'] = rows
        _RUN_STATE['best'] = best
        _RUN_STATE['eval_imgsz_used'] = _eval_imgsz_used

if not ram_stopped_early:
    df_sweep = pd.DataFrame(rows).sort_values(['recall', 'f1'], ascending=False)
    df_sweep.to_csv(ARTIFACT / 'threshold_sweep.csv', index=False)
    print(df_sweep.head(10).round(4).to_string(index=False))

    CONF_BEST = float(best['conf'])
    IOU_BEST = float(best['iou'])
    _det_imgsz_used = _eval_imgsz_used if _eval_imgsz_used is not None else int(IMGSZ_USED)
    det_metrics['best_threshold'] = {'conf': CONF_BEST, 'iou': IOU_BEST, 'f1': best['f1'],
                                     'precision': best.get('precision'), 'recall': best.get('recall'),
                                     'imgsz_used': _det_imgsz_used}
    DET_METRICS_JSON.write_text(json.dumps(det_metrics, indent=2, ensure_ascii=False), encoding='utf-8')
    print(f'CONF_BEST={CONF_BEST} | IOU_BEST={IOU_BEST} | F1={best["f1"]:.4f} | Recall={best.get("recall"):.4f} | imgsz={_det_imgsz_used}')

    # ---- Lưu toàn bộ kết quả để notebook4266 import và chạy tiếp ----
    import numpy as _np
    _eval_checkpoint = {
        'CONF_BEST': CONF_BEST,
        'IOU_BEST': IOU_BEST,
        'IMGSZ_USED': _det_imgsz_used,
        'EVAL_DEVICE': EVAL_DEVICE,
        'EVAL_HALF': EVAL_HALF,
        'TRACK_CONF': float(TRACK_CONF),
        'TRACK_IOU': float(TRACK_IOU),
        'MAX_DET': int(MAX_DET),
        'VAL_CONF': float(VAL_CONF),
        'VAL_IOU': float(VAL_IOU),
        'det_metrics': det_metrics,
        'val_frame_keys': frame_keys,
        'val_image_paths': [str(p) for p in paths],
        'det_metrics_path': str(DET_METRICS_JSON),
        'threshold_sweep_path': str(ARTIFACT / 'threshold_sweep.csv'),
        'ram_stopped': False,
        'stage': 'complete',
    }
    (PRE_EVAL_DIR / 'eval_checkpoint.json').write_text(
        json.dumps(_eval_checkpoint, indent=2, ensure_ascii=False, default=str), encoding='utf-8'
    )
    print('Eval checkpoint saved:', PRE_EVAL_DIR / 'eval_checkpoint.json')

    # Lưu detections val để Cell 7 (notebook4266) không cần chạy lại predict
    _all_boxes_list = []
    _all_confs_list = []
    for _iou_test in [IOU_BEST]:
        for _res in eval_model.predict(
            source=[str(p) for p in paths], conf=0.01, iou=_iou_test,
            imgsz=_det_imgsz_used, max_det=MAX_DET, device=EVAL_DEVICE,
            batch=1, half=EVAL_HALF, stream=True, verbose=False,
        ):
            _b = _res.boxes
            _d = _b.xyxy.cpu().numpy() if _b is not None and len(_b) else np.zeros((0, 4))
            if len(_d):
                _d = np.concatenate([_d, _b.conf.cpu().numpy()[:, None]], 1)
            _all_boxes_list.append(_d[:, :4])
            _all_confs_list.append(_d[:, 4] if _d.shape[1] > 4 else np.zeros(_d.shape[0]))

    if len(_all_boxes_list) > 0:
        _all_boxes = _np.vstack(_all_boxes_list)
        _all_confs = _np.concatenate(_all_confs_list)
    else:
        _all_boxes = np.zeros((0, 4))
        _all_confs = np.zeros((0,))

    _np.savez(
        PRE_EVAL_DIR / 'detection_results.npz',
        boxes=_all_boxes,
        confs=_all_confs,
        frame_keys=np.array(frame_keys, dtype=object),
        image_paths=np.array([str(p) for p in paths], dtype=object),
    )
    print('Detection results saved:', PRE_EVAL_DIR / 'detection_results.npz')

    del eval_model
    gc.collect()
    torch.cuda.empty_cache()
else:
# ── Don RAM/VRAM cuoi Cell 6 ────────────────────────────────
import gc
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    torch.cuda.ipc_collect()
print('Da don RAM/VRAM sau Cell 6 (eval+sweep).')
    print('Cell 6 dừng sớm do vượt ngưỡng RAM. Kết quả trung gian đã lưu để notebook4266 tiếp tục.')


In [14]:
from pathlib import Path
import json

CHECKPOINT_DIR = ARTIFACT / "cell6_checkpoint"

print("Checkpoint directory:", CHECKPOINT_DIR)
print("Exists:", CHECKPOINT_DIR.is_dir())

if CHECKPOINT_DIR.is_dir():
    for p in sorted(CHECKPOINT_DIR.iterdir()):
        print(p.name, "-", p.stat().st_size, "bytes")
required = [
    ARTIFACT / "det_metrics.json",
    ARTIFACT / "threshold_sweep.csv",
    CHECKPOINT_DIR / "detector_checkpoint.json",
    CHECKPOINT_DIR / "threshold_sweep.csv",
]

for path in required:
    print(("OK   " if path.is_file() else "MISS "), path)

Checkpoint directory: /kaggle/working/artifacts/cell6_checkpoint
Exists: True
det_metrics.previous.json - 375 bytes
detector_checkpoint.json - 508 bytes
pre_eval_state.json - 156 bytes
threshold_sweep.csv - 2 bytes
threshold_sweep.partial.csv - 3659 bytes
OK    /kaggle/working/artifacts/det_metrics.json
OK    /kaggle/working/artifacts/threshold_sweep.csv
OK    /kaggle/working/artifacts/cell6_checkpoint/detector_checkpoint.json
OK    /kaggle/working/artifacts/cell6_checkpoint/threshold_sweep.csv


In [15]:
# ── CELL 6B · Lưu checkpoint kết quả detector ─────────────
# Chạy cell này sau Cell 6 để giữ lại kết quả trước khi sang ByteTrack.

CHECKPOINT_DIR = ARTIFACT / 'cell6_checkpoint'
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# Cho phep chay doc lap sau khi kernel restart hoac Cell 6 bi dung som.
_metrics_file = DET_METRICS_JSON if 'DET_METRICS_JSON' in globals() else ARTIFACT / 'det_metrics.json'
_saved_metrics = {}
if _metrics_file.is_file():
    _saved_metrics = json.loads(_metrics_file.read_text(encoding='utf-8'))

_threshold = _saved_metrics.get('best_threshold', {})
_sweep_file = ARTIFACT / 'threshold_sweep.csv'
if not _threshold and _sweep_file.is_file():
    _saved_sweep = pd.read_csv(_sweep_file)
    if not _saved_sweep.empty and 'f1' in _saved_sweep.columns:
        _threshold = _saved_sweep.sort_values('f1', ascending=False).iloc[0].to_dict()

_det_metrics = _saved_metrics
_conf_best = float(_threshold.get('conf', TRACK_CONF))
_iou_best = float(_threshold.get('iou', TRACK_IOU))
_imgsz_used = int(json.loads((CHECKPOINT_DIR / 'detector_checkpoint.json').read_text(encoding='utf-8')).get('IMGSZ_USED', IMGSZ)) if (CHECKPOINT_DIR / 'detector_checkpoint.json').is_file() else int(IMGSZ)
_batch_used = int(json.loads((CHECKPOINT_DIR / 'detector_checkpoint.json').read_text(encoding='utf-8')).get('BATCH_USED', BATCH)) if (CHECKPOINT_DIR / 'detector_checkpoint.json').is_file() else int(BATCH)
checkpoint = {
    'det_metrics': _det_metrics,
    'CONF_BEST': float(_conf_best),
    'IOU_BEST': float(_iou_best),
    'IMGSZ_USED': int(_imgsz_used),
    'BATCH_USED': int(_batch_used),
}
(CHECKPOINT_DIR / 'detector_checkpoint.json').write_text(
    json.dumps(checkpoint, indent=2, ensure_ascii=False), encoding='utf-8'
)

sweep_checkpoint = CHECKPOINT_DIR / 'threshold_sweep.csv'
if 'df_sweep' in globals():
    df_sweep.to_csv(sweep_checkpoint, index=False)
elif _sweep_file.is_file():
    shutil.copy2(_sweep_file, sweep_checkpoint)
else:
    pd.DataFrame([_threshold]).to_csv(sweep_checkpoint, index=False)

print('Đã lưu checkpoint tại:', CHECKPOINT_DIR)
print(' - detector_checkpoint.json')
print(' - threshold_sweep.csv')

# ---- Đóng gói toàn bộ kết quả/model đã có trước thời điểm này ----
# Không đưa dataset BUILD_DIR vào ZIP vì có thể rất lớn; các artifact/model vẫn được giữ.
import zipfile

RESULT_ZIP = WORK / 'visdrone_privacy_results.zip'

def create_results_zip():
    _zip_roots = [ARTIFACT, RUNS_DIR]
    _zip_files = [DATA_YAML, SEQ_INDEX, WORK / 'bytetrack_person.yaml']
    _zip_files = [p for p in _zip_files if p.is_file()]
    with zipfile.ZipFile(RESULT_ZIP, 'w', compression=zipfile.ZIP_DEFLATED, compresslevel=6) as zf:
        for root in _zip_roots:
            if root.is_dir():
                for p in root.rglob('*'):
                    if p.is_file() and p != RESULT_ZIP:
                        zf.write(p, p.relative_to(WORK))
        for p in _zip_files:
            if p != RESULT_ZIP:
                zf.write(p, p.relative_to(WORK))
    return RESULT_ZIP

create_results_zip()

print('Đã đóng gói kết quả/model:', RESULT_ZIP)
print(f'Kích thước ZIP: {RESULT_ZIP.stat().st_size / 2**20:.1f} MiB')

# Chỉ giữ các giá trị nhỏ cần cho Cell 7; xóa model, predictions và bảng lớn khỏi RAM.
for _name in ('eval_model', 'm', 'df_idx', 'df_gt', 'val_idx', 'val_gt', 'gt_map', 'paths',
              'frame_keys', 'rows', 'df_sweep', 'dets', 'counts', 'best', '_saved_sweep'):
    globals().pop(_name, None)
CONF_BEST = _conf_best
IOU_BEST = _iou_best
IMGSZ_USED = _imgsz_used
BATCH_USED = _batch_used
det_metrics = _det_metrics
gc.collect()
if torch.cuda.is_available():
    torch.cuda.empty_cache()
print('Cell 6 memory cleanup complete; Cell 7 uses detector_checkpoint.json and threshold_sweep.csv')


Đã lưu checkpoint tại: /kaggle/working/artifacts/cell6_checkpoint
 - detector_checkpoint.json
 - threshold_sweep.csv
Đã đóng gói kết quả/model: /kaggle/working/visdrone_privacy_results.zip
Kích thước ZIP: 137.0 MiB
Cell 6 memory cleanup complete; Cell 7 uses detector_checkpoint.json and threshold_sweep.csv


In [ ]:
# ── CELL 7/13 · ByteTrack → tracks_frame.csv ───────────────
# Vào : yolo26s.pt, sequence_index.csv
# Ra  : artifacts/tracks_frame.csv

import yaml

TRACKER_YAML = WORK / 'bytetrack_person.yaml'
TRACKER_YAML.write_text(yaml.safe_dump({
    'tracker_type': 'bytetrack',
    'track_high_thresh': 0.35,
    'track_low_thresh': 0.05,
    'new_track_thresh': 0.25,
    'track_buffer': TRACK_BUFFER,
    'match_thresh': 0.7,
    'fuse_score': True,
}, sort_keys=False), encoding='utf-8')

df_idx = pd.read_csv(SEQ_INDEX)
groups = []
for (split, seq), g in df_idx.groupby(['split', 'sequence'], sort=True):
    if split not in TRACK_SPLITS:
        continue
    g = g.sort_values('frame')
    pairs = [(int(r.frame), BUILD_DIR / split / 'images' / r.flat_name,
              int(r.width), int(r.height)) for r in g.itertuples()]
    pairs = [p for p in pairs if p[1].exists()]
    if pairs:
        groups.append((split, seq, pairs))

print('sequence cần track:', len(groups), '| frame:', sum(len(g[2]) for g in groups))

CONF_TRACK = max(float(globals().get('CONF_BEST', TRACK_CONF)), TRACK_CONF)
IOU_TRACK = float(globals().get('IOU_BEST', TRACK_IOU))
print('tracking conf/iou:', CONF_TRACK, IOU_TRACK)

track_model = YOLO(str(BEST_MODEL))
fields = ['split', 'sequence', 'frame', 'track_id', 'class_id', 'confidence',
          'x1', 'y1', 'x2', 'y2', 'width', 'height']
rows = []

for split, seq, pairs in groups:
    track_model.predictor = None
    n = 0
    for res in track_model.track(
        source=[str(p[1]) for p in pairs], persist=True, tracker=str(TRACKER_YAML),
        conf=CONF_TRACK, iou=IOU_TRACK, imgsz=IMGSZ_USED, max_det=MAX_DET,
        device=DEVICE_EVAL, stream=True, verbose=False,
    ):
        frame, _p, iw, ih = pairs[min(n, len(pairs) - 1)]
        n += 1
        b = res.boxes
        if b is None or len(b) == 0:
            continue
        xy = b.xyxy.cpu().numpy()
        cf = b.conf.cpu().numpy()
        cl = b.cls.int().cpu().numpy()
        ids = b.id.int().cpu().numpy() if b.id is not None else -np.ones(len(xy), int)
        for box, s, c, t in zip(xy, cf, cl, ids):
            rows.append({'split': split, 'sequence': seq, 'frame': frame,
                         'track_id': int(t), 'class_id': int(c),
                         'confidence': round(float(s), 6),
                         'x1': round(float(box[0]), 2), 'y1': round(float(box[1]), 2),
                         'x2': round(float(box[2]), 2), 'y2': round(float(box[3]), 2),
                         'width': iw, 'height': ih})
    print(f'  {split:<5} {seq:<24} {n:>4} frame')

df_trk = pd.DataFrame(rows, columns=fields)
df_trk.to_csv(TRACKS_CSV, index=False)
print('\ntracks_frame.csv:', len(df_trk), 'dòng |', df_trk.track_id.nunique(), 'track_id')
print(df_trk.groupby('split').agg(frames=('frame', 'nunique'), rows=('track_id', 'size')).to_string())

del track_model
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# ── CELL 8/13 · Trích 21 đặc trưng mỗi frame ───────────────
# Vào : tracks_frame.csv, sequence_index.csv, gt_boxes.csv
# Ra  : features_frame.csv, feature_schema.json

def crop_gray(img, box):
    if img is None:
        return None
    h, w = img.shape[:2]
    x1, y1, x2, y2 = box
    x1 = int(max(0, math.floor(x1)))
    y1 = int(max(0, math.floor(y1)))
    x2 = int(min(w, math.ceil(x2)))
    y2 = int(min(h, math.ceil(y2)))
    if x2 <= x1 or y2 <= y1:
        return None
    c = img[y1:y2, x1:x2]
    if max(c.shape[:2]) > CROP_RESIZE:
        s = CROP_RESIZE / max(c.shape[:2])
        c = cv2.resize(c, (max(1, int(c.shape[1] * s)), max(1, int(c.shape[0] * s))),
                       interpolation=cv2.INTER_AREA)
    return c


def lap_var(c):
    if c is None or c.size == 0:
        return 0.0
    return float(cv2.Laplacian(c, cv2.CV_64F).var())


df_trk = pd.read_csv(TRACKS_CSV)
df_idx = pd.read_csv(SEQ_INDEX)
df_gt = pd.read_csv(GT_CSV)

meta = {(r.split, r.sequence, int(r.frame)): (int(r.width), int(r.height), r.flat_name)
        for r in df_idx.itertuples()}

train_gt = df_gt[df_gt.split == 'train']
r_ref = float(np.median(train_gt.w / train_gt.h)) if len(train_gt) else 0.4
print('r_ref (median w/h train):', round(r_ref, 4))

rows = []
for (split, seq), g in df_trk.groupby(['split', 'sequence'], sort=True):
    g = g.sort_values(['frame', 'track_id'])
    state = {}
    prev_n = None
    ewma = None

    for frame, fsub in g.groupby('frame', sort=True):
        frame = int(frame)
        iw, ih, flat = meta.get((split, seq, frame), (0, 0, None))
        if iw == 0 or flat is None:
            continue
        D = math.hypot(iw, ih)
        img = cv2.imread(str(BUILD_DIR / split / 'images' / flat), cv2.IMREAD_GRAYSCALE)

        boxes = fsub[['x1', 'y1', 'x2', 'y2']].to_numpy(float)
        N = len(boxes)
        cx = (boxes[:, 0] + boxes[:, 2]) / 2.0
        cy = (boxes[:, 1] + boxes[:, 3]) / 2.0
        centers = np.stack([cx, cy], 1)
        areas = (np.clip(boxes[:, 2] - boxes[:, 0], 1e-6, None) *
                 np.clip(boxes[:, 3] - boxes[:, 1], 1e-6, None))

        dx = centers[:, None, 0] - centers[None, :, 0]
        dy = centers[:, None, 1] - centers[None, :, 1]
        dist = np.hypot(dx, dy)
        np.fill_diagonal(dist, np.inf)
        nearest = np.clip(dist.min(1) / D, 0, 1) if N > 1 else np.ones(N)
        radius = NEIGHBOR_RADIUS_FRAC * D
        n_nb = np.clip((dist <= radius).sum(1) / NEIGHBOR_K, 0, 1) if N > 1 else np.zeros(N)

        x1 = np.maximum(boxes[:, None, 0], boxes[None, :, 0])
        y1 = np.maximum(boxes[:, None, 1], boxes[None, :, 1])
        x2 = np.minimum(boxes[:, None, 2], boxes[None, :, 2])
        y2 = np.minimum(boxes[:, None, 3], boxes[None, :, 3])
        inter = np.clip(x2 - x1, 0, None) * np.clip(y2 - y1, 0, None)
        union = areas[:, None] + areas[None, :] - inter
        iou = np.where(union > 0, inter / union, 0.0)
        np.fill_diagonal(iou, 0.0)
        ov_max = iou.max(1) if N > 1 else np.zeros(N)

        delta = 0.0 if prev_n is None else (N - prev_n) / (prev_n + 1)
        expected = N if ewma is None else ewma
        ratio = N / (expected + 1.0)
        ewma = N if ewma is None else EWMA_ALPHA * N + (1 - EWMA_ALPHA) * ewma
        prev_n = N

        for j, r in enumerate(fsub.itertuples()):
            key = (split, seq, int(r.track_id))
            st = state.get(key)
            if st is None:
                st = {'first': frame, 'prev': None, 'n': 0, 'conf': [], 'cen': [], 'lap': None}
                state[key] = st

            st['n'] += 1
            age = frame - st['first'] + 1
            tsu = 0 if st['prev'] is None else max(0, frame - st['prev'] - 1)
            hit = min(1.0, st['n'] / max(age, 1))

            conf = float(r.confidence)
            c5 = st['conf'][-(CONF_WINDOW - 1):] + [conf] if CONF_WINDOW > 1 else [conf]
            conf_std5 = float(np.std(c5)) if len(c5) > 1 else 0.0
            conf_delta = 0.0 if not st['conf'] else conf - st['conf'][-1]

            bw = max(boxes[j, 2] - boxes[j, 0], 1e-6)
            bh = max(boxes[j, 3] - boxes[j, 1], 1e-6)
            min_side = min(bw, bh)
            aspect = abs(bw / bh - r_ref)
            de = float(max(0.0, min(boxes[j, 0], boxes[j, 1], iw - boxes[j, 2], ih - boxes[j, 3]) / D))
            trunc = 1.0 if min(boxes[j, 0], boxes[j, 1], iw - boxes[j, 2], ih - boxes[j, 3]) <= TRUNC_MARGIN_PX else 0.0

            speed = 0.0
            if st['prev'] is not None:
                speed = float(np.hypot(cx[j] - st['cen'][-1][0], cy[j] - st['cen'][-1][1]) / D)

            cen_win = st['cen'][-(PRED_UNCERT_WINDOW - 1):] + [(cx[j], cy[j])] if PRED_UNCERT_WINDOW > 1 else [(cx[j], cy[j])]
            unc = 0.0
            if len(cen_win) > 1:
                arr = np.array(cen_win)
                unc = float(np.sqrt(arr[:, 0].var() + arr[:, 1].var()) / D)

            crop = crop_gray(img, (boxes[j, 0], boxes[j, 1], boxes[j, 2], boxes[j, 3]))
            lv = lap_var(crop)
            lap_log = float(math.log1p(lv))
            bright = float(crop.mean() / 255.0) if crop is not None and crop.size else 0.0
            blur = 0.0
            if st['lap'] is not None and (st['lap'] + lv) > 0:
                blur = float((st['lap'] - lv) / (st['lap'] + lv + 1e-6))

            tstate = 0.0 if age == 1 else (1.0 if tsu == 0 else 2.0)

            rows.append({
                'split': split, 'sequence': seq, 'frame': frame, 'track_id': int(r.track_id),
                'conf': conf, 'conf_std5': conf_std5, 'conf_delta': conf_delta,
                'track_age': float(age), 'hit_ratio': float(hit),
                'time_since_update': float(tsu), 'track_state': tstate,
                'min_side_px': float(min_side), 'aspect_dev': float(aspect),
                'dist_edge': de, 'is_trunc': trunc,
                'speed_norm': speed, 'pred_uncert': unc,
                'lap_var_log': lap_log, 'brightness': bright, 'motion_blur_ratio': blur,
                'nearest_dist_norm': float(nearest[j]),
                'n_neighbors_r2_norm': float(n_nb[j]),
                'overlap_max': float(ov_max[j]),
                'frame_det_delta': float(delta),
                'n_det_expected_ratio': float(ratio),
            })

            st['prev'] = frame
            st['conf'].append(conf)
            st['cen'].append((float(cx[j]), float(cy[j])))
            st['lap'] = lv

df_feat = repair_tracker_features(pd.DataFrame(rows))
df_feat.to_csv(FEATURES_CSV, index=False)
print('features_frame.csv:', len(df_feat), 'dòng |', len(FEATURES), 'đặc trưng')

schema = {
    'features': FEATURES,
    'count': len(FEATURES),
    'window': {'size': WINDOW_SIZE, 'stride': WINDOW_STRIDE, 'min_frames': MIN_WINDOW_FRAMES},
    'aggregation': {f: ('last' if f in ('conf', 'track_age', 'time_since_update', 'track_state', 'n_det_expected_ratio') else 'mean') for f in FEATURES},
    'definitions': {
        'conf': 'confidence của detection tại frame t',
        'conf_std5': 'std của confidence trong CONF_WINDOW frame gần nhất',
        'conf_delta': 'conf_t - conf_{t-1}',
        'track_age': 't - frame đầu tiên của track + 1',
        'hit_ratio': 'số frame track xuất hiện / track_age',
        'time_since_update': 'số frame trống kể từ lần xuất hiện trước',
        'track_state': '0 tentative, 1 tracked, 2 lost (tái dựng từ chuỗi output ByteTrack)',
        'min_side_px': 'min(w, h) theo pixel',
        'aspect_dev': 'abs(w/h - r_ref), r_ref = median w/h GT train',
        'dist_edge': 'khoảng cách chuẩn hoá tới biên ảnh gần nhất',
        'is_trunc': '1 nếu box chạm biên trong TRUNC_MARGIN_PX',
        'speed_norm': 'dịch chuyển tâm box / đường chéo ảnh',
        'pred_uncert': 'PROXY: std(center) trên PRED_UNCERT_WINDOW frame / D (ByteTrack không expose covariance)',
        'lap_var_log': 'log1p(Var(Laplacian(crop gray)))',
        'brightness': 'mean(crop gray) / 255',
        'motion_blur_ratio': '(L_{t-1} - L_t) / (L_{t-1} + L_t + eps)',
        'nearest_dist_norm': 'khoảng cách tới tâm gần nhất / D',
        'n_neighbors_r2_norm': 'min(số tâm trong bán kính r / K, 1)',
        'overlap_max': 'IoU lớn nhất với box khác trong frame',
        'frame_det_delta': '(N_t - N_{t-1}) / (N_{t-1} + 1)',
        'n_det_expected_ratio': 'N_t / (EWMA(N) + 1), EWMA chỉ dùng quá khứ',
    },
    'notes': [
        'pred_uncert là proxy, không phải trace covariance Kalman.',
        'track_state được tái dựng vì Ultralytics không trả state theo track.',
        'r_ref tính từ GT train, không hard-code.',
    ],
}
SCHEMA_JSON.write_text(json.dumps(schema, indent=2, ensure_ascii=False), encoding='utf-8')
print('feature_schema.json ->', SCHEMA_JSON)
print(df_feat[FEATURES].describe().T[['mean', 'std', 'min', 'max']].round(4).to_string())


In [ ]:
# ── CELL 9/13 · Ẩn danh + tấn công Re-ID độc lập ───────────
# Vào : tracks_frame.csv, gt_boxes.csv, sequence_index.csv
# Ra  : samples.csv, attack_results.csv, track_identity.csv

import torch
import torch.nn.functional as F
from torchvision import transforms


def track_identity(df_trk, df_gt, iou_th):
    gt_map = defaultdict(list)
    for r in df_gt.itertuples():
        gt_map[(r.split, r.sequence, int(r.frame))].append(
            (int(r.target_id), r.x1, r.y1, r.x2, r.y2))
    votes = defaultdict(Counter)
    for (split, seq, frame), sub in df_trk.groupby(['split', 'sequence', 'frame']):
        gts = gt_map.get((split, seq, int(frame)))
        if not gts:
            continue
        tb = sub[['x1', 'y1', 'x2', 'y2']].to_numpy(float)
        gb = np.array([g[1:] for g in gts], float)
        ids = [g[0] for g in gts]
        m = xyxy_iou(tb, gb)
        used_t, used_g = set(), set()
        for flat in np.argsort(-m, axis=None):
            i, j = divmod(int(flat), m.shape[1])
            if m[i, j] < iou_th:
                break
            if i in used_t or j in used_g:
                continue
            used_t.add(i)
            used_g.add(j)
            votes[(split, seq, int(sub.iloc[i].track_id))][ids[j]] += 1

    out = {}
    for k, c in votes.items():
        best, n = c.most_common(1)[0]
        out[k] = (int(best), float(n / sum(c.values())))
    return out


def build_samples(df_trk, ident):
    samples = []
    for (split, seq, tid), sub in df_trk.groupby(['split', 'sequence', 'track_id']):
        frames = np.sort(sub.frame.unique())
        if len(frames) < MIN_WINDOW_FRAMES:
            continue
        if WINDOW_SIZE <= 0:
            wins = [(int(frames[0]), int(frames[-1]))]
        else:
            wins = []
            for s in frames[::WINDOW_STRIDE]:
                e = int(s) + WINDOW_SIZE - 1
                if len(frames[(frames >= s) & (frames <= e)]) < MIN_WINDOW_FRAMES:
                    continue
                wins.append((int(s), e))
                if e >= frames[-1]:
                    break
        info = ident.get((split, seq, int(tid)))
        for s, e in wins:
            samples.append({
                'sample_id': f'{split}|{seq}|{tid}|{s}',
                'split': split, 'sequence': seq, 'track_id': int(tid),
                'window_start': s, 'window_end': e,
                'identity': np.nan if info is None else info[0],
                'identity_match_rate': 0.0 if info is None else info[1],
            })
    return pd.DataFrame(samples)


def build_reid():
    device = f'cuda:{DEVICE_EVAL}' if torch.cuda.is_available() else 'cpu'
    model, name = None, None
    if REID_BACKBONE in ('auto', 'osnet_x1_0'):
        try:
            import torchreid
            model = torchreid.models.build_model(name='osnet_x1_0', num_classes=1000, pretrained=True)
            model.classifier = torch.nn.Identity()
            name = 'osnet_x1_0'
        except Exception as e:
            print('OSNet không khả dụng:', str(e)[:140])
            model = None
    if model is None:
        from torchvision.models import resnet50, ResNet50_Weights
        model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
        model.fc = torch.nn.Identity()
        name = 'resnet50'
    model.eval().to(device)
    tf = transforms.Compose([
        transforms.ToPILImage(),
        transforms.Resize((REID_INPUT, REID_INPUT)),
        transforms.ToTensor(),
        transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225]),
    ])
    print('Re-ID backbone:', name, '| device:', device)
    return model, tf, device, name


def anonymize(img):
    if ANON_METHOD == 'pixelate':
        h, w = img.shape[:2]
        b = max(1, ANON_PIXEL_BLOCK)
        small = cv2.resize(img, (max(1, w // b), max(1, h // b)), interpolation=cv2.INTER_LINEAR)
        return cv2.resize(small, (w, h), interpolation=cv2.INTER_NEAREST)
    k = ANON_KSIZE if ANON_KSIZE % 2 == 1 else ANON_KSIZE + 1
    return cv2.GaussianBlur(img, (k, k), 0)


def crop_color(img, box, pad=0.1):
    if img is None:
        return None
    h, w = img.shape[:2]
    x1, y1, x2, y2 = box
    bw, bh = max(x2 - x1, 1e-6), max(y2 - y1, 1e-6)
    x1 = int(max(0, math.floor(x1 - pad * bw)))
    y1 = int(max(0, math.floor(y1 - pad * bh)))
    x2 = int(min(w, math.ceil(x2 + pad * bw)))
    y2 = int(min(h, math.ceil(y2 + pad * bh)))
    if x2 <= x1 or y2 <= y1:
        return None
    return img[y1:y2, x1:x2]


@torch.no_grad()
def embed_list(model, tf, device, crops):
    if not crops:
        return np.zeros((0, 1), np.float32)
    outs = []
    for i in range(0, len(crops), REID_BATCH):
        batch = torch.stack([tf(c) for c in crops[i:i + REID_BATCH]]).to(device)
        f = F.normalize(model(batch).float(), dim=1)
        outs.append(f.cpu().numpy())
    return np.concatenate(outs, 0)


def build_embedding_cache(df_trk, model, tf, device):
    cache = defaultdict(lambda: {'frames': [], 'clean': [], 'anon': []})
    df_emb = (df_trk.sort_values(['split', 'sequence', 'track_id', 'frame'])
              .assign(_r=lambda d: d.groupby(['split', 'sequence', 'track_id']).cumcount()))
    df_emb = df_emb[df_emb._r % max(1, EMBED_STRIDE) == 0]
    print('crop cần embed:', len(df_emb), '/', len(df_trk))

    for (split, seq, frame), sub in df_emb.groupby(['split', 'sequence', 'frame'], sort=True):
        img = cv2.imread(str(BUILD_DIR / split / 'images' /
                             f'{seq}__{int(frame):07d}.jpg'), cv2.IMREAD_COLOR)
        if img is None:
            continue
        keys, clean, anon = [], [], []
        for r in sub.itertuples():
            c = crop_color(img, (r.x1, r.y1, r.x2, r.y2))
            if c is None or c.size == 0:
                continue
            keys.append((split, seq, int(r.track_id), int(frame)))
            clean.append(c)
            anon.append(anonymize(c))
        if not keys:
            continue
        ec = embed_list(model, tf, device, clean)
        ea = embed_list(model, tf, device, anon)
        for k, a, b in zip(keys, ec, ea):
            d = cache[(k[0], k[1], k[2])]
            d['frames'].append(k[3])
            d['clean'].append(a)
            d['anon'].append(b)

    for k in list(cache):
        d = cache[k]
        order = np.argsort(d['frames'])
        d['frames'] = np.asarray(d['frames'], int)[order]
        d['clean'] = np.stack(d['clean'])[order].astype(np.float32)
        d['anon'] = np.stack(d['anon'])[order].astype(np.float32)
    return dict(cache)


def build_identity_index(cache, ident):
    per = defaultdict(lambda: defaultdict(lambda: {'frames': [], 'embs': []}))
    for (split, seq, tid), d in cache.items():
        info = ident.get((split, seq, tid))
        if info is None or info[1] < LABEL_MIN_MATCH_RATE:
            continue
        slot = per[split][(seq, info[0])]
        slot['frames'].append(d['frames'])
        slot['embs'].append(d['clean'])

    out = {}
    for split, ids in per.items():
        out[split] = {}
        for key, v in ids.items():
            fr = np.concatenate(v['frames'])
            em = np.concatenate(v['embs'])
            order = np.argsort(fr)
            fr, em = fr[order], em[order]
            out[split][key] = {'frames': fr, 'cum': np.cumsum(em, 0), 'n': len(fr)}
    return out


def _unit(v):
    n = float(np.linalg.norm(v))
    return v / n if n > 0 else None


def far_mean(entry, s, e, gap):
    fr, n = entry['frames'], entry['n']
    lo = int(np.searchsorted(fr, s - gap))
    hi = int(np.searchsorted(fr, e + gap + 1))
    n_in = hi - lo
    if n - n_in <= 0:
        return None
    total = entry['cum'][-1]
    insum = entry['cum'][hi - 1] - (entry['cum'][lo - 1] if lo > 0 else 0) if hi > lo else 0
    return _unit((total - insum) / (n - n_in))


def mean_all(entry):
    return _unit(entry['cum'][-1] / entry['n'])


df_trk = pd.read_csv(TRACKS_CSV)
df_gt = pd.read_csv(GT_CSV)

IDENT = track_identity(df_trk, df_gt, TRACK_ID_IOU)
df_tid = pd.DataFrame([{'split': k[0], 'sequence': k[1], 'track_id': k[2],
                        'identity': v[0], 'match_rate': v[1]} for k, v in IDENT.items()])
df_tid.to_csv(ARTIFACT / 'track_identity.csv', index=False)

df_samples = build_samples(df_trk, IDENT)
df_samples.to_csv(SAMPLES_CSV, index=False)
print('samples:', len(df_samples), '| có identity:',
      int(df_samples.identity.notna().sum()))

model, tf, device, reid_name = build_reid()
cache = build_embedding_cache(df_trk, model, tf, device)
print('crop đã embed:', len(cache), 'track')
id_index = build_identity_index(cache, IDENT)
print('identity index:', {k: len(v) for k, v in id_index.items()})

rng = np.random.RandomState(SEED)
attack = []
for r in df_samples.itertuples():
    rec = {'sample_id': r.sample_id, 'split': r.split, 'sequence': r.sequence,
           'track_id': int(r.track_id), 'window_start': int(r.window_start),
           'window_end': int(r.window_end), 'identity_true': r.identity,
           'n_probe': 0, 'n_gallery_ids': 0, 'top1_identity': np.nan,
           'top1_sim': np.nan, 'true_sim': np.nan, 'rank_true': np.nan,
           'top1_correct': np.nan, 'status': 'ok'}

    if pd.isna(r.identity) or r.identity_match_rate < LABEL_MIN_MATCH_RATE:
        rec['status'] = 'no_identity'
        attack.append(rec)
        continue

    d = cache.get((r.split, r.sequence, int(r.track_id)))
    if d is None:
        rec['status'] = 'no_probe'
        attack.append(rec)
        continue

    sel = np.where((d['frames'] >= r.window_start) & (d['frames'] <= r.window_end))[0]
    if len(sel) == 0:
        rec['status'] = 'no_probe'
        attack.append(rec)
        continue
    sel = sel[::PROBE_STRIDE][:MAX_PROBE_CROPS]
    probe = _unit(d['anon'][sel].mean(0))
    if probe is None:
        rec['status'] = 'no_probe'
        attack.append(rec)
        continue
    rec['n_probe'] = int(len(sel))

    candidates = id_index.get(r.split, {})
    keys = list(candidates.keys())
    if GALLERY_CROSS_SEQUENCE:
        pool = keys
    else:
        pool = [k for k in keys if k[0] == r.sequence]
    true_key = (r.sequence, int(r.identity))
    if true_key not in pool:
        rec['status'] = 'no_gallery'
        attack.append(rec)
        continue
    if len(pool) > MAX_GALLERY_IDS:
        others = [k for k in pool if k != true_key]
        pick = rng.choice(len(others), size=min(len(others), MAX_GALLERY_IDS - 1), replace=False)
        pool = [true_key] + [others[i] for i in sorted(pick)]

    sims = {}
    for key in pool:
        entry = candidates[key]
        if key == true_key:
            g = far_mean(entry, r.window_start, r.window_end, GALLERY_MIN_GAP)
        else:
            g = mean_all(entry)
        if g is not None:
            sims[key] = float(probe @ g)

    if true_key not in sims:
        rec['status'] = 'no_gallery'
        attack.append(rec)
        continue
    if len(sims) < MIN_GALLERY_IDS:
        rec['status'] = 'too_few_ids'
        attack.append(rec)
        continue

    order = sorted(sims.items(), key=lambda kv: -kv[1])
    top1_key, top1_sim = order[0]
    rec['n_gallery_ids'] = len(sims)
    rec['top1_identity'] = top1_key[1]
    rec['top1_sim'] = top1_sim
    rec['true_sim'] = sims[true_key]
    rec['rank_true'] = float([k for k, _ in order].index(true_key) + 1)
    rec['top1_correct'] = int(top1_key == true_key)
    attack.append(rec)

df_att = pd.DataFrame(attack)
df_att.to_csv(ATTACK_CSV, index=False)

print('\nattack_results.csv:', len(df_att), 'dòng')
print(df_att.status.value_counts().to_string())
ok = df_att[df_att.status == 'ok']
if len(ok):
    print('top-1 đúng:', f'{ok.top1_correct.mean():.3f}',
          '| sim trung bình:', f'{ok.top1_sim.mean():.3f}')

(ARTIFACT / 'attack_meta.json').write_text(json.dumps({
    'backbone': reid_name, 'anon_method': ANON_METHOD,
    'anon_ksize': ANON_KSIZE, 'anon_block': ANON_PIXEL_BLOCK,
    'gallery_min_gap': GALLERY_MIN_GAP, 'cross_sequence': GALLERY_CROSS_SEQUENCE,
    'window': [WINDOW_SIZE, WINDOW_STRIDE], 'features': len(FEATURES),
}, indent=2), encoding='utf-8')

del model
gc.collect()
torch.cuda.empty_cache()


In [ ]:
# ── CELL 10/13 · Chọn ngưỡng τ trên val (Youden J) ─────────
# Vào : attack_results.csv
# Ra  : tau.json, roc.png

import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, roc_auc_score

df_att = pd.read_csv(ATTACK_CSV)
val = df_att[(df_att.split == TAU_SELECT_SPLIT) & (df_att.status == 'ok')].copy()
print('mẫu val dùng chọn τ:', len(val))

tau_info = {'tau': 0.5, 'youden_j': None, 'auc': None,
            'n_val': int(len(val)), 'split': TAU_SELECT_SPLIT}

if len(val) and val.top1_correct.nunique() > 1:
    y = val.top1_correct.astype(int).values
    score = val.top1_sim.values
    fpr, tpr, thr = roc_curve(y, score)
    j = tpr - fpr
    k = int(np.argmax(j))
    tau = float(np.clip(thr[k], 0.0, 1.0))
    if not np.isfinite(tau):
        tau = 0.5
    tau_info.update({'tau': tau, 'youden_j': float(j[k]),
                     'auc': float(roc_auc_score(y, score)),
                     'tpr': float(tpr[k]), 'fpr': float(fpr[k])})

    fig, ax = plt.subplots(figsize=(5, 5))
    ax.plot(fpr, tpr, label=f'AUC = {tau_info["auc"]:.3f}')
    ax.plot([0, 1], [0, 1], '--', color='gray', lw=1)
    ax.scatter([fpr[k]], [tpr[k]], color='red', zorder=3,
               label=f'τ = {tau:.3f} (J = {j[k]:.3f})')
    ax.set_xlabel('False Positive Rate')
    ax.set_ylabel('True Positive Rate')
    ax.set_title(f'Re-ID attack ROC trên {TAU_SELECT_SPLIT}')
    ax.legend(loc='lower right')
    fig.tight_layout()
    fig.savefig(ARTIFACT / 'roc.png', dpi=140)
    plt.close(fig)
else:
    raise RuntimeError('Validation split must contain both attack classes; refusing arbitrary tau=0.5')

TAU_JSON.write_text(json.dumps(tau_info, indent=2, ensure_ascii=False), encoding='utf-8')
TAU = float(tau_info['tau'])
print(json.dumps(tau_info, indent=2, ensure_ascii=False))


In [ ]:
# ── CELL 11/13 · Gán nhãn + gộp track-level → risk_dataset.csv ──
# Vào : features_frame.csv, samples.csv, attack_results.csv, tau.json
# Ra  : risk_dataset.csv (21 đặc trưng + label)
# SỬA: CỬA SỔ ĐẶC TRUNG PHẢI CẢUSAL (không dùng frame > window_start / cutoff).

import math
df_feat = pd.read_csv(FEATURES_CSV)
df_samples = pd.read_csv(SAMPLES_CSV)
df_att = pd.read_csv(ATTACK_CSV)
TAU = float(json.loads(TAU_JSON.read_text(encoding="utf-8"))["tau"])

AGG_LAST = {"conf", "track_age", "time_since_update", "track_state", "n_det_expected_ratio"}
att_map = {r.sample_id: r for r in df_att.itertuples()}

feat_groups = {k: g.sort_values("frame")
               for k, g in df_feat.groupby(["split", "sequence", "track_id"])}

records, skipped = [], Counter()
for r in df_samples.itertuples():
    g = feat_groups.get((r.split, r.sequence, int(r.track_id)))
    if g is None:
        skipped["no_features"] += 1
        continue

    # ---- CẢUSAL: chỉ dùng frame <= window_start (không bao giờ nhìn tương lai) ----
    cutoff = int(r.window_start)
    sub = g[g.frame <= cutoff]
    if len(sub) < MIN_WINDOW_FRAMES:
        skipped["short_window"] += 1
        continue

    rec = {
        "sample_id": r.sample_id, "split": r.split, "sequence": r.sequence,
        "track_id": int(r.track_id), "window_start": int(r.window_start),
        "window_end": int(r.window_end), "cutoff": cutoff, "identity": r.identity,
        "n_frames": int(len(sub)),
    }
    for f in FEATURES:
        rec[f] = float(sub[f].iloc[-1]) if f in AGG_LAST else float(sub[f].mean())

    a = att_map.get(r.sample_id)
    if a is None or a.status != "ok" or pd.isna(a.identity_true):
        rec["label"] = np.nan
        rec["label_reason"] = "unreliable" if a is None else a.status
    else:
        rec["top1_identity"] = a.top1_identity
        rec["top1_sim"] = a.top1_sim
        rec["true_sim"] = a.true_sim
        rec["rank_true"] = a.rank_true
        rec["label_reason"] = "attack"
        if int(a.top1_correct) == 1 and a.top1_sim >= TAU:
            rec["label"] = 1.0
        else:
            rec["label"] = 0.0
    records.append(rec)

df_risk = pd.DataFrame(records)
df_risk.to_csv(RISK_CSV, index=False)

print("risk_dataset.csv:", len(df_risk), "mẫu |", len(FEATURES), "đặc trưng")
print("bỏ qua:", dict(skipped))
print("\nphân bố nhãn theo split:")
print(df_risk.groupby("split").label.agg(["count", "sum", "mean"]).round(3).to_string())
print("\nnhãn tổng:", df_risk.label.value_counts(dropna=False).to_dict())
for f in FEATURES:
    assert f in df_risk.columns, f
print("\n21 cột đặc trưng:", FEATURES)


In [ ]:
# ── CELL 12/13 · Train XGBoost (GroupKFold theo sequence) ──
# Vào : risk_dataset.csv
# Ra  : risk_model.json, risk_model_calibrated.joblib, risk_oof.csv, cv_summary.json

import xgboost as xgb
import joblib
from sklearn.model_selection import GroupKFold
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import roc_auc_score, average_precision_score, brier_score_loss

df = pd.read_csv(RISK_CSV)
lab = df[df.label.notna()].copy()
lab["label"] = lab.label.astype(int)
print("mẫu có nhãn:", len(lab), "| dương:", int(lab.label.sum()),
      "| âm:", int((1 - lab.label).sum()))
print("sequence:", lab.sequence.nunique())

X = lab[FEATURES].to_numpy(np.float32)
y = lab.label.to_numpy(int)
groups = lab.sequence.to_numpy()
n_groups = len(np.unique(groups))
can_cv = (len(np.unique(y)) > 1) and (n_groups >= 2)
n_splits = max(2, min(N_FOLDS, n_groups)) if can_cv else 0

params = dict(XGB_PARAMS)
if 0 < y.sum() < len(y):
    params["scale_pos_weight"] = float((1 - y.mean()) / max(y.mean(), 1e-6))

oof = np.full(len(lab), np.nan)
fold_models, cv_rows = [], []

if not can_cv:
    print("CẢNH BÁO: chỉ có 1 lớp hoặc <2 sequence → không thể CV, dùng prior làm baseline.")
    oof[:] = float(y.mean())
    cv_summary = {"n_samples": int(len(lab)), "n_groups": int(n_groups),
                  "degenerate": True, "prior": float(y.mean())}
else:
    gkf = GroupKFold(n_splits=n_splits)
    for fold, (tr, va) in enumerate(gkf.split(X, y, groups)):
        model = xgb.XGBClassifier(**params)
        model.fit(X[tr], y[tr], eval_set=[(X[va], y[va])], verbose=False)
        oof[va] = model.predict_proba(X[va])[:, 1]
        auc = roc_auc_score(y[va], oof[va]) if len(np.unique(y[va])) > 1 else np.nan
        ap = average_precision_score(y[va], oof[va]) if len(np.unique(y[va])) > 1 else np.nan
        cv_rows.append({"fold": fold, "n_train": int(len(tr)), "n_val": int(len(va)),
                        "auc": None if np.isnan(auc) else float(auc),
                        "pr_auc": None if np.isnan(ap) else float(ap),
                        "best_iteration": int(getattr(model, "best_iteration", 0) or 0)})
        fold_models.append(model)
        print(f"  fold {fold}: n_tr={len(tr)} n_va={len(va)} AUC={auc:.4f} PR-AUC={ap:.4f}")

    cv_summary = {"n_samples": int(len(lab)), "n_groups": int(n_groups),
                  "n_splits": int(n_splits), "degenerate": False,
                  "folds": cv_rows,
                  "oof_auc": float(roc_auc_score(y, oof)),
                  "oof_pr_auc": float(average_precision_score(y, oof))}

# ---- Huấn luyện model cuối trên toàn bộ dữ liệu có nhãn ----
final_model = xgb.XGBClassifier(**{k: v for k, v in params.items()
                                   if k != "early_stopping_rounds"})
final_model.fit(X, y, verbose=False)
final_model.save_model(str(RISK_MODEL_JSON))
save_model_metadata(RISK_MODEL_JSON, FEATURES)
print("risk_model.json ->", RISK_MODEL_JSON)

# ---- Calibration (Platt scaling / Isotonic) trên OOF ----
if len(np.unique(y)) > 1 and len(lab) > 50:
    # Dùng OOF predictions làm input cho calibration (không rò rỉ vì OOF là out-of-fold)
    cal = CalibratedClassifierCV(final_model, method="isotonic", cv="prefit")
    cal.fit(X, y)
    p_raw = final_model.predict_proba(X)[:, 1]
    p_cal = cal.predict_proba(X)[:, 1]

    brier_raw = float(brier_score_loss(y, p_raw))
    brier_cal = float(brier_score_loss(y, p_cal))
    print(f"Brier trước calibration: {brier_raw:.4f} | sau calibration: {brier_cal:.4f}")

    # Lưu model đã calibration
    joblib.dump(cal, ARTIFACT / "risk_model_calibrated.joblib")
    print("risk_model_calibrated.joblib ->", ARTIFACT / "risk_model_calibrated.joblib")

    cv_summary["brier_raw"] = brier_raw
    cv_summary["brier_calibrated"] = brier_cal
    cv_summary["calibration_method"] = "isotonic_prefit_on_oof"
else:
    cv_summary["calibration_method"] = "skipped_insufficient_samples"

pd.DataFrame({"sample_id": lab.sample_id, "split": lab.split,
              "sequence": lab.sequence, "y_true": y, "y_prob": oof}).to_csv(RISK_OOF_CSV, index=False)
(ARTIFACT / "cv_summary.json").write_text(json.dumps(cv_summary, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps({k: v for k, v in cv_summary.items() if k != "folds"}, indent=2))


In [ ]:
# ── CELL 13/13 · Đo hiệu năng + SHAP + export ──────────────
# Vào : risk_model.json, risk_model_calibrated.joblib, risk_oof.csv, risk_dataset.csv
# Ra  : risk_metrics.json, calib.png, importance.png, shap.png, zip

from sklearn.metrics import (roc_auc_score, average_precision_score, f1_score,
                             precision_score, recall_score, brier_score_loss,
                             roc_curve, confusion_matrix)
from sklearn.calibration import calibration_curve, expected_calibration_error
from sklearn.linear_model import LogisticRegression
import joblib
import matplotlib.pyplot as plt

df = pd.read_csv(RISK_CSV)
lab = df[df.label.notna()].copy()
lab["label"] = lab.label.astype(int)
oof = pd.read_csv(RISK_OOF_CSV)
load_model_metadata(RISK_MODEL_JSON)

X = lab[FEATURES].to_numpy(np.float32)
y = lab.label.to_numpy(int)
p_raw = oof.y_prob.to_numpy(float)

# ---- Thử tải model đã calibration; nếu không có, dùng raw ----
cal_path = ARTIFACT / "risk_model_calibrated.joblib"
p_cal = None
cal_used = False
if cal_path.is_file():
    try:
        cal = joblib.load(cal_path)
        p_cal = cal.predict_proba(X)[:, 1]
        cal_used = True
        print("Dùng model đã calibration (isotonic).")
    except Exception as e:
        print("Không load được model calibration:", str(e)[:160])

p = p_cal if cal_used else p_raw

# ---- chọn ngưỡng quyết định trên OUF (calibrated nếu có) ----
thr_info = {"threshold": RISK_THRESHOLD, "method": "fixed",
            "calibrated": bool(cal_used)}
if len(np.unique(y)) > 1:
    fpr, tpr, thr = roc_curve(y, p)
    j = tpr - fpr
    k = int(np.argmax(j))
    thr_info = {"threshold": float(np.clip(thr[k], 0, 1)), "method": "youden_j_oof",
                "j": float(j[k]), "calibrated": bool(cal_used)}
thr = thr_info["threshold"]
y_hat = (p >= thr).astype(int)

def expected_calibration_error(y_true, prob, bins=10):
    edges = np.linspace(0.0, 1.0, bins + 1)
    ece = 0.0
    for lo, hi in zip(edges[:-1], edges[1:]):
        mask = (prob >= lo) & (prob < hi if hi < 1 else prob <= hi)
        if mask.any():
            ece += float(mask.mean()) * abs(float(prob[mask].mean()) - float(y_true[mask].mean()))
    return ece

metrics = {
    "n_samples": int(len(lab)),
    "n_positive": int(y.sum()),
    "positive_rate": float(y.mean()),
    "threshold": thr_info,
    "auc": float(roc_auc_score(y, p)) if len(np.unique(y)) > 1 else None,
    "pr_auc": float(average_precision_score(y, p)) if len(np.unique(y)) > 1 else None,
    "f1": float(f1_score(y, y_hat, zero_division=0)),
    "precision": float(precision_score(y, y_hat, zero_division=0)),
    "recall": float(recall_score(y, y_hat, zero_division=0)),
    "brier": float(brier_score_loss(y, p)) if len(np.unique(y)) > 1 else None,
    "ece": expected_calibration_error(y, p) if len(np.unique(y)) > 1 else None,
    "calibration_used": bool(cal_used),
    "evaluation_protocol": "sequence-grouped OOF over all labelled sequences; causal features (no look-ahead)",
    "confusion_matrix": confusion_matrix(y, y_hat).tolist(),
}

# ---- baseline so sánh ----
baselines = {}
if len(np.unique(y)) > 1:
    prior = np.full(len(y), y.mean())
    baselines["prior"] = {"brier": float(brier_score_loss(y, prior)),
                          "auc": 0.5}
    lr = LogisticRegression(max_iter=2000, class_weight="balanced")
    lr.fit(X, y)
    lp = lr.predict_proba(X)[:, 1]
    baselines["logreg_in_sample"] = {"auc": float(roc_auc_score(y, lp)),
                                     "brier": float(brier_score_loss(y, lp))}
    conf_idx = FEATURES.index("conf")
    cp = X[:, conf_idx]
    baselines["conf_threshold"] = {"auc": float(roc_auc_score(y, cp))}
    metrics["baselines"] = baselines

# ---- calibration curve ----
if len(np.unique(y)) > 1:
    try:
        frac_pos, mean_pred = calibration_curve(y, p, n_bins=10, strategy="quantile")
        fig, ax = plt.subplots(figsize=(5, 5))
        label = "XGBoost (calibrated OOF)" if cal_used else "XGBoost (raw OOF)"
        ax.plot(mean_pred, frac_pos, "o-", label=label)
        ax.plot([0, 1], [0, 1], "--", color="gray", lw=1)
        ax.set_xlabel("Xác suất dự đoán")
        ax.set_ylabel("Tỷ lệ dương thực tế")
        ax.set_title(f"Calibration | Brier = {metrics['brier']:.4f} | ECE = {metrics['ece']:.4f}")
        ax.legend()
        fig.tight_layout()
        fig.savefig(ARTIFACT / "calib.png", dpi=140)
        plt.close(fig)
    except Exception as e:
        print("Calibration lỗi:", str(e)[:160])

# ---- feature importance ----
model = xgb.XGBClassifier()
model.load_model(str(RISK_MODEL_JSON))
gain_map = model.get_booster().get_score(importance_type="gain")
imp = pd.DataFrame({
    "feature": FEATURES,
    "gain": [float(gain_map.get(f"f{i}", gain_map.get(f, 0.0))) for i, f in enumerate(FEATURES)],
})
imp = imp.sort_values("gain", ascending=False)
imp.to_csv(ARTIFACT / "feature_importance.csv", index=False)

fig, ax = plt.subplots(figsize=(6, 6))
ax.barh(imp.feature[::-1], imp.gain[::-1], color="steelblue")
ax.set_xlabel("Gain")
ax.set_title("XGBoost feature importance")
fig.tight_layout()
fig.savefig(ARTIFACT / "importance.png", dpi=140)
plt.close(fig)

# ---- SHAP (tuỳ chọn) ----
try:
    import shap
    explainer = shap.TreeExplainer(model)
    sample_n = min(500, len(X))
    sv = explainer.shap_values(X[:sample_n])
    fig = plt.figure(figsize=(8, 6))
    shap.summary_plot(sv, X[:sample_n], feature_names=FEATURES, show=False)
    fig.tight_layout()
    fig.savefig(ARTIFACT / "shap.png", dpi=140, bbox_inches="tight")
    plt.close(fig)
    metrics["shap"] = "shap.png"
except Exception as e:
    print("SHAP không khả dụng:", str(e)[:160])
    metrics["shap"] = None

# ---- export detector ----
if EXPORT_ONNX:
    try:
        exp = YOLO(str(BEST_MODEL))
        onnx_path = Path(exp.export(format="onnx", imgsz=IMGSZ_USED, opset=ONNX_OPSET,
                                    dynamic=False, simplify=True, half=False))
        shutil.copy2(onnx_path, ARTIFACT / onnx_path.name)
        print("ONNX ->", ARTIFACT / onnx_path.name)
        del exp
        gc.collect()
        torch.cuda.empty_cache()
    except Exception as e:
        print("Export ONNX lỗi:", str(e)[:160])

shutil.copy2(BEST_MODEL, ARTIFACT / "yolo26s.pt")
shutil.copy2(DATA_YAML, ARTIFACT / "data_person.yaml")

RISK_METRICS_JSON.write_text(json.dumps(metrics, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps({k: v for k, v in metrics.items() if k != "confusion_matrix"},
                 indent=2, ensure_ascii=False))

zip_path = create_results_zip() if "create_results_zip" in globals() else WORK / "visdrone_privacy_results.zip"
print("\nfile ZIP tổng ->", zip_path, f"({Path(zip_path).stat().st_size / 2**20:.1f} MiB)")
print("artifact:", sorted(p.name for p in ARTIFACT.iterdir()))
